# 🐻‍❄️ Polars Library:

Polars is one of the fastest and most modern DataFrame libraries in Python for data science and analysis.

🌟 **What is Polars?**

Polars is an open-source DataFrame library written in Rust, with a Python API, developed for data analysis and manipulation. It is built on the Apache Arrow columnar memory format, executes operations on multiple CPU cores in parallel, and offers both an eager API and a lazy (query-optimized) API.

**Basic Data Structures:**
1. Series:

    A one-dimensional, typed data structure with a **name**.

    Can be thought of as a single column.

    Polars Series has **no index**; elements are addressed by position.

2. DataFrame:

    A two-dimensional data structure with named, typed columns.

    Similar to a table or Excel/SQL database structure.

    Rows have **no index/labels**, they are addressed by position only. If you need a label, keep it in a regular column.

    It is the most used data structure in Polars.

3. LazyFrame:

    A DataFrame-like *query plan*. Nothing is computed until you call `.collect()`, which lets Polars optimize the whole query (predicate/projection pushdown, parallelism, streaming).

4. Expressions (`pl.col(...)`):

    The building block of the Polars API. An expression describes a computation (e.g. `pl.col("Salary") * 12`) and is evaluated inside contexts such as `select`, `with_columns`, `filter` and `group_by`.

📈 **Why Use Polars?**

Polars is designed to overcome performance and memory problems encountered when working with large, real-world data.

* Data Loading and Reading: It can load data from different formats (CSV, Parquet, Excel, SQL, JSON/NDJSON, Arrow IPC, etc.) into a DataFrame; `scan_*` functions read lazily, so only the required rows/columns are loaded.

* Data Cleaning:

    * Detecting, filling, or deleting missing values (`null`; note that `NaN` is a separate concept in Polars).

    * Converting data types (`cast`, converting text to numbers). Polars is strictly typed.

    * Removing duplicate rows (`unique`).

* Data Manipulation and Transformation:

    * Filtering, selecting, and slicing data (`filter`, `select`, `slice`).

    * Adding/removing columns, deleting rows (`with_columns`, `drop`).

    * Grouping data (`group_by`) and aggregating (mean, sum, count, etc.).

    * Merging, appending, or joining datasets (`join`, `concat`).

* Data Analysis: Performing statistical summaries and analyses, calculating correlations.

* Performance: Multi-threaded execution, columnar memory layout, lazy evaluation and a query optimizer.

🤝 **Relationship with NumPy and Other Libraries**

Polars works well with the rest of the Python data science ecosystem, but it is a **standalone engine**, not a layer on top of NumPy.

🐘 1 . **Relationship with NumPy**

* NumPy (Numerical Python) is the base Python library for high-performance numerical calculations.

* Basic Structure: Unlike Pandas, Polars is **not** built on NumPy. Data is stored in Apache Arrow compatible columnar buffers managed by the Rust engine.

* Interoperability: `pl.Series(np_array)` / `pl.DataFrame(np_array)` create Polars objects from arrays, and `.to_numpy()` converts back (zero-copy when the data allows it, e.g. numeric columns without nulls). NumPy ufuncs (`np.log(series)`) also work on Polars Series.

🖼️ 2 . Relationship with Matplotlib and Seaborn (Visualization)

* Role: After cleaning and preparing data with Polars, you can visualize it with these libraries.

* Integration: Many plotting libraries (Altair, Plotly, recent Seaborn versions) accept Polars DataFrames directly. For Matplotlib, pass columns as arrays (`df["col"].to_numpy()`), or convert with `df.to_pandas()` (requires `pyarrow`).

🤖 3 . Relationship with Scikit-learn and Deep Learning Libraries

* Role: Polars performs the data preparation and preprocessing steps required for Machine Learning (ML).

* Flow: Data is cleaned and transformed with Polars; when it is ready for modeling, it is converted into NumPy arrays (`df.to_numpy()`) and presented as input to Scikit-learn or deep learning libraries (TensorFlow, PyTorch). Recent Scikit-learn versions can also return Polars DataFrames from transformers via `set_output(transform="polars")`.

🐼 4 . Relationship with Pandas

* `df.to_pandas()` and `pl.from_pandas(pdf)` make it easy to move between the two libraries (requires `pyarrow`), so you can adopt Polars gradually.

| Library | Focus | Data Structure | Role |
| :--- | :--- | :--- | :--- |
| **NumPy** | Numerical Computing | Multi-dimensional Arrays | Basic mathematical operations and speed. |
| **Polars** | Data Manipulation | DataFrame, LazyFrame and Series | Data reading, cleaning, and preparation (fast, multi-threaded). |
| **Matplotlib/Seaborn** | Data Visualization | Graphs and Plots | Visualizing analysis results. |
| **Scikit-learn** | Machine Learning | Models, Algorithms | Creating prediction models. |

# Installation and requirements


Polars installation
```
uv add polars
```

To read Excel files via Polars (the default engine)

```
uv add fastexcel
```

To write Excel files via Polars

```
uv add xlsxwriter
```

Optional: for converting to/from Pandas (`to_pandas()`, `pl.from_pandas()`)

```
uv add pandas pyarrow
```

In [1]:
# If polars is not installed: (run in terminal)
# pip install polars fastexcel xlsxwriter

import polars as pl  # Standard alias
import polars.selectors as cs  # Column selectors (select columns by dtype, name pattern, ...)
import numpy as np  # We will use in some examples
import time
from datetime import date

# Pandas → Polars Cheat Sheet

The lesson below is written for Polars. If you come from Pandas, these are the most common translations:

| Pandas | Polars |
| :--- | :--- |
| `pd.DataFrame(data)` | `pl.DataFrame(data)` |
| `df["a"]` | `df["a"]` (Series) or `pl.col("a")` (expression) |
| `df[["a", "b"]]` | `df.select("a", "b")` |
| `df[df["a"] > 3]` | `df.filter(pl.col("a") > 3)` |
| `df["c"] = df["a"] * 2` | `df = df.with_columns((pl.col("a") * 2).alias("c"))` |
| `df.drop("c", axis=1)` | `df.drop("c")` |
| `df.groupby("k")["v"].mean()` | `df.group_by("k").agg(pl.col("v").mean())` |
| `df.sort_values("a", ascending=False)` | `df.sort("a", descending=True)` |
| `df.isnull().sum()` | `df.null_count()` |
| `df.dropna()` / `df.fillna(0)` | `df.drop_nulls()` / `df.fill_null(0)` |
| `pd.merge(a, b, on="k", how="left")` | `a.join(b, on="k", how="left")` |
| `pd.concat([a, b], ignore_index=True)` | `pl.concat([a, b])` |
| `df.loc[...]` / `df.iloc[...]` | `filter(...)` / `df[...]` / `df.row(i)` (no index) |
| `df.reset_index()` | `df.with_row_index()` |
| `df.set_index("k")` | not needed, keep `k` as a column |
| `df.query("a > 3")` | `df.filter(pl.col("a") > 3)` or `df.sql(...)` |
| `df.astype({"a": "float64"})` | `df.cast({"a": pl.Float64})` |
| `df.pivot_table(...)` | `df.pivot(on=..., index=..., values=..., aggregate_function=...)` |
| `df.apply(...)` / `s.apply(...)` | `map_elements(...)` (slow, prefer expressions) |
| `df.info()` | `df.glimpse()` / `df.schema` |
| `df.describe(include="all")` | `df.describe()` (already includes all columns) |
| `NaN` for missing values | `null` for missing values (`NaN` is a separate float value) |
| in-place modification | Polars is immutable: every operation returns a new object |

# Polars Series

A single-column, typed array with a name. It holds data and a name, but **no index**.
Examples with different types (int, float, string, date).

If you need labels for your values (what Pandas keeps in the index), keep them in a separate column of a DataFrame.

In [2]:
# 1) Creating a simple Series (using a regular list)
s = pl.Series("s", [10, 20, 30, 40])
# s is a Series object; it has a name ("s") and a dtype (Int64), but no index
print("\n#1 - s:\n", s)
# Explanation: Output shows; name, dtype, length + values


#1 - s:
 shape: (4,)
Series: 's' [i64]
[
	10
	20
	30
	40
]


In [3]:
# 2) "Labeled" values: Polars has no index, so labels live in their own column
s2 = pl.DataFrame({"label": ["a", "b", "c"], "value": [100, 200, 300]})
print("\n#2 - s2:\n", s2)
# Selecting by label: filter on the label column
print('Printing "b" from s2 as table:\n', s2.filter(pl.col("label") == "b"))

print('Printing "b" value:', s2.filter(pl.col("label") == "b")["value"].item())  # 200


#2 - s2:
 shape: (3, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 100   │
│ b     ┆ 200   │
│ c     ┆ 300   │
└───────┴───────┘
Printing "b" from s2 as table:
 shape: (1, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ b     ┆ 200   │
└───────┴───────┘
Printing "b" value: 200


In [4]:
# 3) Creating from Dictionary (keys become one column, values another)
data = {"istanbul": 15, "ankara": 5, "izmir": 8}
pop = pl.DataFrame({"city": list(data.keys()), "population": list(data.values())})

# Alternative: create a DataFrame from a list of tuples (key, value)
# pop = pl.DataFrame(list(data.items()), schema=["city", "population"], orient='row')  # same as above

# Another alternative: create a DataFrame from zipped keys and values
# pop = pl.DataFrame(zip(data.keys(), data.values()), schema=["city", "population"]) 

print("\n#3 - pop\n", pop)

# If we query, a non-existent label returns an EMPTY DataFrame (no NaN, no KeyError):
print(pop.filter(pl.col("city") == "antalya"))

# For plain key lookups, a regular dict is still the right tool:
print(data.get("antalya"))  # None


#3 - pop
 shape: (3, 2)
┌──────────┬────────────┐
│ city     ┆ population │
│ ---      ┆ ---        │
│ str      ┆ i64        │
╞══════════╪════════════╡
│ istanbul ┆ 15         │
│ ankara   ┆ 5          │
│ izmir    ┆ 8          │
└──────────┴────────────┘
shape: (0, 2)
┌──────┬────────────┐
│ city ┆ population │
│ ---  ┆ ---        │
│ str  ┆ i64        │
╞══════╪════════════╡
└──────┴────────────┘
None


In [5]:
# 4) Mathematical operation with Series (broadcast)
s3 = pl.Series("s3", [1, 2, 3])
print("\n#4 - Mathematical operations:\n", s3 * 10)  # all values multiplied by 10


#4 - Mathematical operations:
 shape: (3,)
Series: 's3' [i64]
[
	10
	20
	30
]


In [6]:
# 5) Checking Series type and name
print(s3.dtype)  # Int64
print(s3.name)  # s3

# Series is a typed, Arrow-backed column. Unlike Pandas there is no index to align on:
# operations between two Series are POSITIONAL.

Int64
s3


# Creating Series with Numpy Array

In [7]:
np_array = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
series = pl.Series("numbers", np_array)
print(series)

print('Type of series:', type(series))

print(
    '\n Series properties: \n',
    '\n Series data type: ', series.dtype,
    '\n Series shape: ', series.shape,
    '\n Series length: ', series.len(),
    '\n Series name: ', series.name,
    '\n Series data: ', series.to_numpy(), # gets the data part as a NumPy array
    '\n Series list:',  series.to_list(),  # ... or as a Python list
)

print("\nlists first 5 elements: \n", series.head(5))  # Series.head() default is 10
print("\nlists first 2 elements:\n", series.head(2))  # lists first 2 elements

print("\nlists last 5 elements: \n", series.tail(5))  # Series.tail() default is 10
print("\nlists last 2 elements:\n", series.tail(2))  # lists last 2 elements

shape: (10,)
Series: 'numbers' [i64]
[
	1
	2
	3
	4
	5
	6
	7
	8
	9
	10
]
Type of series: <class 'polars.series.series.Series'>

 Series properties: 
 
 Series data type:  Int64 
 Series shape:  (10,) 
 Series length:  10 
 Series name:  numbers 
 Series data:  [ 1  2  3  4  5  6  7  8  9 10] 
 Series list: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

lists first 5 elements: 
 shape: (5,)
Series: 'numbers' [i64]
[
	1
	2
	3
	4
	5
]

lists first 2 elements:
 shape: (2,)
Series: 'numbers' [i64]
[
	1
	2
]

lists last 5 elements: 
 shape: (5,)
Series: 'numbers' [i64]
[
	6
	7
	8
	9
	10
]

lists last 2 elements:
 shape: (2,)
Series: 'numbers' [i64]
[
	9
	10
]


# Indexing Operations in Series

Polars has no index, so there are two distinct ways to access data:

* **Positional access** (`series[i]`, slices, `gather`) works directly on a Series.
* **Label-based access** (what `.loc` does in Pandas) is done with `filter` on a label column of a DataFrame.

In [8]:
# Creating Series and a labeled DataFrame
labels = ["a", "b", "c", "d", "e"]
series = pl.Series("series", [10, 20, 30, 40, 50])
df_lbl = pl.DataFrame({"label": labels, "value": series})

print(series)
print(df_lbl)

shape: (5,)
Series: 'series' [i64]
[
	10
	20
	30
	40
	50
]
shape: (5, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 10    │
│ b     ┆ 20    │
│ c     ┆ 30    │
│ d     ┆ 40    │
│ e     ┆ 50    │
└───────┴───────┘


# Basic Indexing in Series

In [9]:
# Accessing a single value (by position)
print(series[1])  # 20

# Accessing multiple values (list of positions)
print('\nMultiple values:\n', series[[0, 2, 4]])

# Slice operation (positional, end-exclusive)
print('\nSlice operations:\n', series[1:4])

# Accessing by label: filter on the label column
print('\nFiltering operations:\n', df_lbl.filter(pl.col("label").is_in(["a", "c", "e"])))

20

Multiple values:
 shape: (3,)
Series: 'series' [i64]
[
	10
	30
	50
]

Slice operations:
 shape: (3,)
Series: 'series' [i64]
[
	20
	30
	40
]

Filtering operations:
 shape: (3, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 10    │
│ c     ┆ 30    │
│ e     ┆ 50    │
└───────┴───────┘


# Label-based Indexing (the `.loc` equivalent)

Pandas' `.loc` becomes `filter` on the label column.

In [10]:
# Single label
print('\nSingle value:\n',df_lbl.filter(pl.col("label") == "c")["value"].item())  # 30

# Multiple labels
print('\nMultiple values:\n', df_lbl.filter(pl.col("label").is_in(["a", "d"])))

# Range of labels (inclusive on both ends, like .loc slices)
print('\nRange of label:\n', df_lbl.filter(pl.col("label").is_between(pl.lit("b"), pl.lit("d"))))


Single value:
 30

Multiple values:
 shape: (2, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 10    │
│ d     ┆ 40    │
└───────┴───────┘

Range of label:
 shape: (3, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ b     ┆ 20    │
│ c     ┆ 30    │
│ d     ┆ 40    │
└───────┴───────┘


# Position-based Indexing (the `.iloc` equivalent)

In [11]:
# Position-based indexing
# Single position
print('\nSingle position: #1\n', series[2])  # 30
print('\nSingle position: #2\n', series.item(2))  # same, explicit scalar access

# Multiple positions
print('\nMultiple operations:\n', series.gather([0, 3, 4]))
# Slice operation (end-exclusive)

print('\nSlice operations: #1\n', series[1:4])
print('\nSlice operations: #2\n', 
    series.slice(1, 3)
)  # slice(offset, length) -> same result, also works lazily on DataFrames


Single position: #1
 30

Single position: #2
 30

Multiple operations:
 shape: (3,)
Series: 'series' [i64]
[
	10
	40
	50
]

Slice operations: #1
 shape: (3,)
Series: 'series' [i64]
[
	20
	30
	40
]

Slice operations: #2
 shape: (3,)
Series: 'series' [i64]
[
	20
	30
	40
]


# Conditional Operations in Series

In [12]:
# Filtering by condition
series = pl.Series("series", [10, 25, 30, 45, 50])
df_lbl = pl.DataFrame({"label": ["a", "b", "c", "d", "e"], "value": series})

# Values greater than 30
print('\nGreater than 30:\n', series.filter(series > 30))

# Multiple conditions
print('\nMultiple conditions:\n', series.filter((series > 20) & (series < 50)))

# Equality to specific values
print('\nEquality to specific values:\n', series.is_in([10, 30, 50]))
print('\nFiltered by specific values:#1\n', series.filter(series.is_in([10, 30, 50])))

# Same thing on the labeled DataFrame (keeps the labels)
print('\nFiltered by specific values:#2\n', df_lbl.filter(pl.col("value").is_in([10, 30, 50])))


Greater than 30:
 shape: (2,)
Series: 'series' [i64]
[
	45
	50
]

Multiple conditions:
 shape: (3,)
Series: 'series' [i64]
[
	25
	30
	45
]

Equality to specific values:
 shape: (5,)
Series: 'series' [bool]
[
	true
	false
	true
	false
	true
]

Filtered by specific values:#1
 shape: (3,)
Series: 'series' [i64]
[
	10
	30
	50
]

Filtered by specific values:#2
 shape: (3, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 10    │
│ c     ┆ 30    │
│ e     ┆ 50    │
└───────┴───────┘


# Creating Multi-Level Index

In [13]:
# Multi-level "index" as ordinary columns
multi = pl.DataFrame(
    {
        "letter": ["A", "A", "B", "B"],
        "number": [1, 2, 1, 2],
        "value": [10, 20, 30, 40],
    }
)
print('\nMulti-level DataFrame:\n', multi)

# Select the first level
print('\nFiltered by first level:\n', multi.filter(pl.col("letter") == "A"))

# Select a specific (letter, number) pair
print('\nFiltered by specific (letter, number) pair:\n', 
    multi.filter((pl.col("letter") == "A") & (pl.col("number") == 2))["value"].item()
)  # 20

# Cross-section (like .xs): rows where letter == "B", without the letter column
print('\nCross-section (like .xs): rows where letter == "B", without the letter column:\n', multi.filter(pl.col("letter") == "B").drop("letter"))

# Aggregating over a level
print('\nAggregation over a level:\n', multi.group_by("letter").agg(pl.col("value").sum()).sort("letter"))


Multi-level DataFrame:
 shape: (4, 3)
┌────────┬────────┬───────┐
│ letter ┆ number ┆ value │
│ ---    ┆ ---    ┆ ---   │
│ str    ┆ i64    ┆ i64   │
╞════════╪════════╪═══════╡
│ A      ┆ 1      ┆ 10    │
│ A      ┆ 2      ┆ 20    │
│ B      ┆ 1      ┆ 30    │
│ B      ┆ 2      ┆ 40    │
└────────┴────────┴───────┘

Filtered by first level:
 shape: (2, 3)
┌────────┬────────┬───────┐
│ letter ┆ number ┆ value │
│ ---    ┆ ---    ┆ ---   │
│ str    ┆ i64    ┆ i64   │
╞════════╪════════╪═══════╡
│ A      ┆ 1      ┆ 10    │
│ A      ┆ 2      ┆ 20    │
└────────┴────────┴───────┘

Filtered by specific (letter, number) pair:
 20

Cross-section (like .xs): rows where letter == "B", without the letter column:
 shape: (2, 2)
┌────────┬───────┐
│ number ┆ value │
│ ---    ┆ ---   │
│ i64    ┆ i64   │
╞════════╪═══════╡
│ 1      ┆ 30    │
│ 2      ┆ 40    │
└────────┴───────┘

Aggregation over a level:
 shape: (2, 2)
┌────────┬───────┐
│ letter ┆ value │
│ ---    ┆ ---   │
│ str    ┆ i64   │
╞═

# Creating Time Series

In [14]:
# Time series: a Date column + a value column
dates = pl.date_range(date(2023, 1, 1), date(2023, 1, 5), interval="1d", eager=True)
ts = pl.DataFrame({"date": dates, "value": [100, 110, 105, 115, 120]})

print('\nTime Series:\n', ts)

# Date-based filtering
print('\nDate-based filtering:\n', ts.filter(pl.col("date") == date(2023, 1, 3))["value"].item())  # 105
print('\nDate-based filtering (range):\n', ts.filter(pl.col("date").is_between(date(2023, 1, 2), date(2023, 1, 4))))


Time Series:
 shape: (5, 2)
┌────────────┬───────┐
│ date       ┆ value │
│ ---        ┆ ---   │
│ date       ┆ i64   │
╞════════════╪═══════╡
│ 2023-01-01 ┆ 100   │
│ 2023-01-02 ┆ 110   │
│ 2023-01-03 ┆ 105   │
│ 2023-01-04 ┆ 115   │
│ 2023-01-05 ┆ 120   │
└────────────┴───────┘

Date-based filtering:
 105

Date-based filtering (range):
 shape: (3, 2)
┌────────────┬───────┐
│ date       ┆ value │
│ ---        ┆ ---   │
│ date       ┆ i64   │
╞════════════╪═══════╡
│ 2023-01-02 ┆ 110   │
│ 2023-01-03 ┆ 105   │
│ 2023-01-04 ┆ 115   │
└────────────┴───────┘


# Changing Index

Since there is no index to change, "changing the index" means adding, renaming or removing a label column.

In [15]:
# "changing index"
series = pl.Series("values", [1, 2, 3, 4, 5])
print(series)

# Assigning labels: they become a normal column
df_series = pl.DataFrame({"label": ["x", "y", "z", "t", "w"], "values": series})
print(df_series)

# reset_index() equivalent: with_row_index() adds a 0 - n-1 counter column (UInt32)
print(df_series.with_row_index())
print(df_series.with_row_index(name="id", offset=1))  # custom name and starting offset

# set_index() equivalent: not needed, the label column already is the "key".
# For fast repeated lookups by key, convert to a dict:
lookup = dict(df_series.iter_rows())
print(lookup["z"])  # 3

shape: (5,)
Series: 'values' [i64]
[
	1
	2
	3
	4
	5
]
shape: (5, 2)
┌───────┬────────┐
│ label ┆ values │
│ ---   ┆ ---    │
│ str   ┆ i64    │
╞═══════╪════════╡
│ x     ┆ 1      │
│ y     ┆ 2      │
│ z     ┆ 3      │
│ t     ┆ 4      │
│ w     ┆ 5      │
└───────┴────────┘
shape: (5, 3)
┌───────┬───────┬────────┐
│ index ┆ label ┆ values │
│ ---   ┆ ---   ┆ ---    │
│ u32   ┆ str   ┆ i64    │
╞═══════╪═══════╪════════╡
│ 0     ┆ x     ┆ 1      │
│ 1     ┆ y     ┆ 2      │
│ 2     ┆ z     ┆ 3      │
│ 3     ┆ t     ┆ 4      │
│ 4     ┆ w     ┆ 5      │
└───────┴───────┴────────┘
shape: (5, 3)
┌─────┬───────┬────────┐
│ id  ┆ label ┆ values │
│ --- ┆ ---   ┆ ---    │
│ u32 ┆ str   ┆ i64    │
╞═════╪═══════╪════════╡
│ 1   ┆ x     ┆ 1      │
│ 2   ┆ y     ┆ 2      │
│ 3   ┆ z     ┆ 3      │
│ 4   ┆ t     ┆ 4      │
│ 5   ┆ w     ┆ 5      │
└─────┴───────┴────────┘
3


# Querying Labels

In [16]:
# label querying operations
df_series = pl.DataFrame({"label": ["a", "b", "c"], "value": [10, 20, 30]})

print('\nSeries column values:\n', df_series.columns)  # ['label', 'value']

print('\nSeries label values:\n', df_series["label"].to_list())  # ['a', 'b', 'c']

print('\nIs "b" in the label column?\n', "b" in df_series["label"])  # True

print('\nData type of the label column:\n', df_series["label"].dtype)  # String

print('\nRows of the DataFrame:\n', df_series.rows())  # list of tuples: [('a', 10), ('b', 20), ('c', 30)]

print('\nIterator of the DataFrame:\n', list(df_series.iter_rows()))  # same, as an iterator

print('\nDictionary of the DataFrame:\n', dict(df_series.iter_rows()))  # {'a': 10, 'b': 20, 'c': 30}


Series column values:
 ['label', 'value']

Series label values:
 ['a', 'b', 'c']

Is "b" in the label column?
 True

Data type of the label column:
 String

Rows of the DataFrame:
 [('a', 10), ('b', 20), ('c', 30)]

Iterator of the DataFrame:
 [('a', 10), ('b', 20), ('c', 30)]

Dictionary of the DataFrame:
 {'a': 10, 'b': 20, 'c': 30}


# Missing Labels

In [17]:
# looking up a label that does not exist
df_series = pl.DataFrame({"label": ["a", "b", "c", "d"], "value": [1, 2, 3, 4]})

# Non-existent label -> empty result; .item() on an empty result raises ValueError
try:
    print(df_series.filter(pl.col("label") == "x")["value"].item())
except ValueError as e:
    print(f"Error: {e}")


# Safe access with a default
def get_value(df, label, default="Not Found"):
    found = df.filter(pl.col("label") == label)["value"]
    return found[0] if len(found) else default


print(get_value(df_series, "x"))  # Not Found
print(get_value(df_series, "a"))  # 1

# or use a dict and .get()
lookup = dict(df_series.iter_rows())
print(lookup.get("x", "Not Found"))  # Not Found
print(lookup.get("a", "Not Found"))  # 1


Error: can only call '.item()' if the Series is of length 1, or an explicit index is provided (Series is of length 0)
Not Found
1
Not Found
1


# Sorting Series and Sorting by Label

In [18]:
# Unsorted labels
df_series = pl.DataFrame({"label": ["d", "a", "c", "b"], "value": [1, 2, 3, 4]})
print('\nUnsorted dataframe:\n', df_series)

# Sorting by label (sort_index equivalent)
print('\nSorted dataframe by label:\n', df_series.sort("label"))

# Sorting a plain Series by value
print('\nSorted dataframe by values:\n', df_series["value"].sort(descending=True))


Unsorted dataframe:
 shape: (4, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ d     ┆ 1     │
│ a     ┆ 2     │
│ c     ┆ 3     │
│ b     ┆ 4     │
└───────┴───────┘

Sorted dataframe by label:
 shape: (4, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ a     ┆ 2     │
│ b     ┆ 4     │
│ c     ┆ 3     │
│ d     ┆ 1     │
└───────┴───────┘

Sorted dataframe by values:
 shape: (4,)
Series: 'value' [i64]
[
	4
	3
	2
	1
]


# Conditional Filtering Example: Performance Comparison

Let's compare different ways of filtering a large Series. Timings change from machine to machine and from run to run
(the first run includes warm-up), so run the cell several times.

In [19]:
# Let's create a large series
large_series = pl.Series("v", range(1_000_000))
large_df = large_series.to_frame()

print(f"Series size: {len(large_series)}")
print('\nMeasuring performance of different indexing methods on large series...')

# 1. Series filter with a boolean mask
start = time.perf_counter()
result1 = large_series.filter(large_series > 5_000).head(10)
duration1 = time.perf_counter() - start
print(f"1. Series.filter duration:        {duration1:.6f} seconds")

# 2. Eager DataFrame with expressions
start = time.perf_counter()
result2 = large_df.filter(pl.col("v") > 5_000).head(10)
duration2 = time.perf_counter() - start
print(f"2. DataFrame.filter duration:     {duration2:.6f} seconds")

# 3. Lazy query: the optimizer sees filter + head together
start = time.perf_counter()
result3 = large_df.lazy().filter(pl.col("v") > 5_000).head(10).collect()
duration3 = time.perf_counter() - start
print(f"3. LazyFrame duration:            {duration3:.6f} seconds")

# 4. Pure numpy
start = time.perf_counter()
values = large_series.to_numpy()
result4 = values[values > 5_000][:10]
duration4 = time.perf_counter() - start
print(f"4. Pure numpy method duration:    {duration4:.6f} seconds")

print("\nSpeed comparison:")
print(f'Numpy is {duration4/duration1:.1f} times slower than Series.filter')
print(f'Numpy is {duration4/duration2:.1f} times slower than DataFrame.filter')
print(f'Numpy is {duration4/duration3:.1f} times slower than LazyFrame')

Series size: 1000000

Measuring performance of different indexing methods on large series...
1. Series.filter duration:        0.000203 seconds
2. DataFrame.filter duration:     0.000587 seconds
3. LazyFrame duration:            0.000683 seconds
4. Pure numpy method duration:    0.000923 seconds

Speed comparison:
Numpy is 4.6 times slower than Series.filter
Numpy is 1.6 times slower than DataFrame.filter
Numpy is 1.4 times slower than LazyFrame


In [20]:
# Let's examine memory usage and data types
print("\nData structure analysis:")
print(f"Polars Series type: {type(large_series)}")
print(f"Numpy array type: {type(large_series.to_numpy())}")
print(f"Polars Series dtype: {large_series.dtype}")
print(f"Polars Series size: {large_series.estimated_size()} bytes (no index is stored)")
print(f"Numpy values size: {large_series.to_numpy().nbytes} bytes")

# Operation details
print("\nOperation complexity:")
print("1. Series.filter: boolean mask → filter → new Series")
print(
    "2. DataFrame.filter: expression evaluated by the Rust engine (multi-threaded) → new DataFrame"
)
print(
    "3. LazyFrame: query plan optimized (e.g. predicate pushdown) → executed once on .collect()"
)
print("4. Numpy: Pure array operation → Array slice → Array")


Data structure analysis:
Polars Series type: <class 'polars.series.series.Series'>
Numpy array type: <class 'numpy.ndarray'>
Polars Series dtype: Int64
Polars Series size: 8000000 bytes (no index is stored)
Numpy values size: 8000000 bytes

Operation complexity:
1. Series.filter: boolean mask → filter → new Series
2. DataFrame.filter: expression evaluated by the Rust engine (multi-threaded) → new DataFrame
3. LazyFrame: query plan optimized (e.g. predicate pushdown) → executed once on .collect()
4. Numpy: Pure array operation → Array slice → Array


In [21]:
# Let's do a more realistic test
print("\n" + "=" * 50)
print("REALISTIC SCENARIO: More complex filter")
print("=" * 50)

# Series filled with random numbers
np.random.seed(42)
random_values = np.random.randint(0, 1_000_000, 1_000_000)
realistic_series = pl.Series("v", random_values)
realistic_df = realistic_series.to_frame()

# 1. Series filter
start = time.perf_counter()
result1 = realistic_series.filter(
    (realistic_series > 5_000) & (realistic_series % 2 == 0)
).head(20)
duration1 = time.perf_counter() - start

# 2. Eager DataFrame
start = time.perf_counter()
result2 = realistic_df.filter((pl.col("v") > 5_000) & (pl.col("v") % 2 == 0)).head(20)
duration2 = time.perf_counter() - start

# 3. Lazy
start = time.perf_counter()
result3 = (
    realistic_df.lazy()
    .filter((pl.col("v") > 5_000) & (pl.col("v") % 2 == 0))
    .head(20)
    .collect()
)
duration3 = time.perf_counter() - start

# 4. Numpy
start = time.perf_counter()
values = realistic_series.to_numpy()
mask = (values > 5_000) & (values % 2 == 0)
result4 = values[mask][:20]
duration4 = time.perf_counter() - start

print(f"Series.filter:    {duration1:.6f}s")
print(f"DataFrame.filter: {duration2:.6f}s")
print(f"LazyFrame:        {duration3:.6f}s")
print(f"Numpy:            {duration4:.6f}s")
print(
    f"All four methods give the same result: {result1.to_list() == result2['v'].to_list() == result3['v'].to_list() == result4.tolist()}"
)
print(f'Numpy is {duration4/duration1:.1f} times slower than Series.filter')
print(f'Numpy is {duration4/duration2:.1f} times slower than DataFrame.filter')
print(f'Numpy is {duration4/duration3:.1f} times slower than LazyFrame')



REALISTIC SCENARIO: More complex filter
Series.filter:    0.004005s
DataFrame.filter: 0.001250s
LazyFrame:        0.001134s
Numpy:            0.005561s
All four methods give the same result: True
Numpy is 1.4 times slower than Series.filter
Numpy is 4.4 times slower than DataFrame.filter
Numpy is 4.9 times slower than LazyFrame


# Series Information

* Series = Named, typed, one-dimensional data structure
* Can be created with list, dictionary values, or NumPy array.
* Has **no index**: elements are addressed by position. Labels, if needed, live in a separate column.
* Offers features like mathematical operations, filtering, missing data (null) management.
* It is one of the building blocks of Polars (a DataFrame is a collection of equal-length Series).

# Concatenating Series
Combining multiple Polars Series into a single Series.
This operation is usually done vertically (one after another).

Think of it like:

Series 1: [10, 20]
Series 2: [30, 40]
When combined: [10, 20, 30, 40]

The `pl.concat()` function is used to combine Series (and DataFrames/LazyFrames) in Polars.

`pl.concat([series1, series2, series3, ...])`

Since there is no index, there is nothing to keep or reset: the result is always a fresh, positional Series.

In [22]:
series1 = pl.Series("x", [1, 2])
series2 = pl.Series("x", [3, 4])

combined = pl.concat([series1, series2])
print(combined)

shape: (4,)
Series: 'x' [i64]
[
	1
	2
	3
	4
]


**Combining Series with Same Labels**
Since Polars has no index, duplicate labels are not a concern for Series. If you keep labels in a column,
concatenating DataFrames simply stacks the rows (duplicate labels are allowed):

In [23]:
series1 = pl.DataFrame({"label": ["x", "y"], "value": [10, 20]})
series2 = pl.DataFrame({"label": ["x", "y"], "value": [30, 40]})

result = pl.concat([series1, series2])
print('Combined DataFrames:', result)

# Plain Series: no index → no need for ignore_index / reset_index(drop=True)
result = pl.concat([series1["value"], series2["value"]])
# Output: 10, 20, 30, 40
print('\nCombined Series (no index):', result)

Combined DataFrames: shape: (4, 2)
┌───────┬───────┐
│ label ┆ value │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ x     ┆ 10    │
│ y     ┆ 20    │
│ x     ┆ 30    │
│ y     ┆ 40    │
└───────┴───────┘

Combined Series (no index): shape: (4,)
Series: 'value' [i64]
[
	10
	20
	30
	40
]


**Combining Multiple Series**

In [24]:
s1 = pl.Series("v", [1])
s2 = pl.Series("v", [2])
s3 = pl.Series("v", [3])
s4 = pl.Series("v", [4])

all_series = pl.concat([s1, s2, s3, s4])
print(all_series)

shape: (4,)
Series: 'v' [i64]
[
	1
	2
	3
	4
]


# Combining Series Side-by-Side

Series are single-column, so combining them side-by-side creates a DataFrame.

In [25]:
series1 = pl.Series("A", [1, 2])
series2 = pl.Series("B", [3, 4])
series3 = pl.Series("C", [5, 6])

df = pl.DataFrame([series1, series2, series3])
print('Combined DataFrame:\n', df)
# Note: Naming series is necessary for column names (names must be unique in a DataFrame).

# Alternative: horizontal concat of DataFrames
df_h = pl.concat(
    [series1.to_frame(), series2.to_frame(), series3.to_frame()], how="horizontal"
)
print('\nCombined DataFrame (horizontal):\n', df_h)

# how="vertical"   → stack rows (default)
# how="horizontal" → place side by side as columns (equal heights required)
# how="horizontal_extend" → side by side, shorter frames padded with null
# how="diagonal"   → stack rows, union of columns (missing filled with null)

Combined DataFrame:
 shape: (2, 3)
┌─────┬─────┬─────┐
│ A   ┆ B   ┆ C   │
│ --- ┆ --- ┆ --- │
│ i64 ┆ i64 ┆ i64 │
╞═════╪═════╪═════╡
│ 1   ┆ 3   ┆ 5   │
│ 2   ┆ 4   ┆ 6   │
└─────┴─────┴─────┘

Combined DataFrame (horizontal):
 shape: (2, 3)
┌─────┬─────┬─────┐
│ A   ┆ B   ┆ C   │
│ --- ┆ --- ┆ --- │
│ i64 ┆ i64 ┆ i64 │
╞═════╪═════╪═════╡
│ 1   ┆ 3   ┆ 5   │
│ 2   ┆ 4   ┆ 6   │
└─────┴─────┴─────┘


# Combining with Missing Data

Polars does not align on an index. Side-by-side concatenation is **positional**: with `how="horizontal_extend"` shorter inputs are padded with `null` at the end (plain `how="horizontal"` raises a `ShapeError` for different heights).
If you want to align by label, use a `join`.

In [26]:
s1 = pl.DataFrame({"label": ["a", "b"], "X": [1, 2]})
s2 = pl.DataFrame({"label": ["a"], "Y": [3]})

# Positional: how="horizontal_extend" pads the shorter frame with null at the bottom
# (how="horizontal" raises a ShapeError for different heights)
df = pl.concat([s1.select("X"), s2.select("Y")], how="horizontal_extend")
print('Positional concatenation (with null padding):\n', df)

# Label-aligned (what pd.concat(axis=1) did with the index)
df = s1.join(s2, on="label", how="full", coalesce=True).sort("label")
print('\nLabel-aligned concatenation:\n', df)

Positional concatenation (with null padding):
 shape: (2, 2)
┌─────┬──────┐
│ X   ┆ Y    │
│ --- ┆ ---  │
│ i64 ┆ i64  │
╞═════╪══════╡
│ 1   ┆ 3    │
│ 2   ┆ null │
└─────┴──────┘

Label-aligned concatenation:
 shape: (2, 3)
┌───────┬─────┬──────┐
│ label ┆ X   ┆ Y    │
│ ---   ┆ --- ┆ ---  │
│ str   ┆ i64 ┆ i64  │
╞═══════╪═════╪══════╡
│ a     ┆ 1   ┆ 3    │
│ b     ┆ 2   ┆ null │
└───────┴─────┴──────┘


***Practical Tip***
If you want to read many series from a file and combine them

```
series_list = [series1, series2, series3, ...]
all_data = pl.concat(series_list)
```
This is very common in data collection operations.
To read many files at once, `pl.scan_csv("data/*.csv")` or `pl.read_csv(...)` + `pl.concat` is the usual pattern.

# Arithmetic Operations in Series

Arithmetic operations in Polars Series are applied to each element individually.

For example:

* series * 2 → Multiplies every number in the Series by 2.
* series1 + series2 → Adds elements at the same **position**.

These operations are **positional**: there is no index alignment. Both Series must have the same length
(or one of them must have length 1, which is broadcast). Polars raises an error for any other length mismatch.
If you want to align by label, `join` the data first.

In [27]:
prices = pl.DataFrame({"item": ["apple", "pear", "banana"], "price": [10, 20, 30]})
print("Original:\n", prices)

# Let's add 10% raise → i.e., multiply each price by 1.1
new_price = prices['price']*1.1
print("\nPrices with raise:\n", new_price)

# Add a new column with the raised prices
new_prices = prices.with_columns((pl.col("price") * 1.1).alias("new_price"))
print("\nPrices with raise:\n", new_prices)

Original:
 shape: (3, 2)
┌────────┬───────┐
│ item   ┆ price │
│ ---    ┆ ---   │
│ str    ┆ i64   │
╞════════╪═══════╡
│ apple  ┆ 10    │
│ pear   ┆ 20    │
│ banana ┆ 30    │
└────────┴───────┘

Prices with raise:
 shape: (3,)
Series: 'price' [f64]
[
	11.0
	22.0
	33.0
]

Prices with raise:
 shape: (3, 3)
┌────────┬───────┬───────────┐
│ item   ┆ price ┆ new_price │
│ ---    ┆ ---   ┆ ---       │
│ str    ┆ i64   ┆ f64       │
╞════════╪═══════╪═══════════╡
│ apple  ┆ 10    ┆ 11.0      │
│ pear   ┆ 20    ┆ 22.0      │
│ banana ┆ 30    ┆ 33.0      │
└────────┴───────┴───────────┘


In [28]:
# Let's create example series
series1 = pl.Series("series1", [10, 20, 30, 40, 50])
series2 = pl.Series("series2", [1, 2, 3, 4, 5])

print("Series1 (to Numpy):", series1.to_numpy())
print("Series2 (to Numpy):", series2.to_numpy())

# Addition
total = series1 + series2
print("Total:", total.to_numpy())

# Subtraction
difference = series1 - series2
print("Difference:", difference.to_numpy())

# Multiplication
product = series1 * series2
print("Product:", product.to_numpy())

# Division (true division → Float64; use // for floor division)
division = series1 / series2
print("Division:", division.to_numpy())

# Power
power = series1**2
print("Square of Series1:", power.to_numpy())

Series1 (to Numpy): [10 20 30 40 50]
Series2 (to Numpy): [1 2 3 4 5]
Total: [11 22 33 44 55]
Difference: [ 9 18 27 36 45]
Product: [ 10  40  90 160 250]
Division: [10. 10. 10. 10. 10.]
Square of Series1: [ 100  400  900 1600 2500]


In [29]:
# Series with constant number operations Scalar operations
print("Series1 + 100:", (series1 + 100).to_numpy())
print("Series1 * 2:", (series1 * 2).to_numpy())
print("Series1 / 10:", (series1 / 10).to_numpy())
print("Series1 ** 0.5:", (series1**0.5).to_numpy())  # Square root

Series1 + 100: [110 120 130 140 150]
Series1 * 2: [ 20  40  60  80 100]
Series1 / 10: [1. 2. 3. 4. 5.]
Series1 ** 0.5: [3.16227766 4.47213595 5.47722558 6.32455532 7.07106781]


In [30]:
# math functions: Polars has native methods ...
print(series1)
print("Logarithm:", series1.log().to_numpy())  # natural log
print("Exponential:", (series1 / 10).exp().to_numpy())
print("Square root:", series1.sqrt().to_numpy())
print("Absolute value:", pl.Series([-5, -2, 0, 3, 7]).abs().to_numpy())
print("Sine:", series1.sin().to_numpy())
print("Rounding:", (series1 / 3).round(2).to_numpy())

# ... and NumPy ufuncs also work on Polars Series (the result is a Polars Series)
print("\nnp.log on a Polars Series:", type(np.log(series1)))

shape: (5,)
Series: 'series1' [i64]
[
	10
	20
	30
	40
	50
]
Logarithm: [2.30258509 2.99573227 3.40119738 3.68887945 3.91202301]
Exponential: [  2.71828183   7.3890561   20.08553692  54.59815003 148.4131591 ]
Square root: [3.16227766 4.47213595 5.47722558 6.32455532 7.07106781]
Absolute value: [5 2 0 3 7]
Sine: [-0.54402111  0.91294525 -0.98803162  0.74511316 -0.26237485]
Rounding: [ 3.33  6.67 10.   13.33 16.67]

np.log on a Polars Series: <class 'polars.series.series.Series'>


# Operations with Differently Labeled Series

Pandas aligns on the index automatically and puts `NaN` where labels don't match. Polars never does this: Series
operations are positional. To reproduce label alignment, `join` on the label column.

In [31]:
# Data with different labels
series_a = pl.DataFrame({"label": ["a", "b", "c"], "a": [10, 20, 30]})
series_b = pl.DataFrame({"label": ["b", "c", "d"], "b": [5, 15, 25]})

print("Series A:\n", series_a)
print("\nSeries B:\n", series_b)

# Positional operations require equal length (or length 1):
try:
    print(pl.Series([10, 20, 30]) + pl.Series([5, 15]))
except Exception as e:
    print(f"\n{type(e).__name__}: {e}")

# Label alignment via full join: non-common labels get null
joined = series_a.join(series_b, on="label", how="full", coalesce=True).sort("label")
print("\nSeries A + Series B:")
total_mismatch = joined.with_columns((pl.col("a") + pl.col("b")).alias("a+b"))
print(total_mismatch)
print("null values occurred for non-common labels!")

Series A:
 shape: (3, 2)
┌───────┬─────┐
│ label ┆ a   │
│ ---   ┆ --- │
│ str   ┆ i64 │
╞═══════╪═════╡
│ a     ┆ 10  │
│ b     ┆ 20  │
│ c     ┆ 30  │
└───────┴─────┘

Series B:
 shape: (3, 2)
┌───────┬─────┐
│ label ┆ b   │
│ ---   ┆ --- │
│ str   ┆ i64 │
╞═══════╪═════╡
│ b     ┆ 5   │
│ c     ┆ 15  │
│ d     ┆ 25  │
└───────┴─────┘

InvalidOperationError: cannot do arithmetic operation on series of different lengths: got 3 and 2

Series A + Series B:
shape: (4, 4)
┌───────┬──────┬──────┬──────┐
│ label ┆ a    ┆ b    ┆ a+b  │
│ ---   ┆ ---  ┆ ---  ┆ ---  │
│ str   ┆ i64  ┆ i64  ┆ i64  │
╞═══════╪══════╪══════╪══════╡
│ a     ┆ 10   ┆ null ┆ null │
│ b     ┆ 20   ┆ 5    ┆ 25   │
│ c     ┆ 30   ┆ 15   ┆ 45   │
│ d     ┆ null ┆ 25   ┆ null │
└───────┴──────┴──────┴──────┘
null values occurred for non-common labels!


In [32]:
# Managing null values
print("\nAdding by setting null values to 0:")
total_fillna = joined.with_columns(
    (pl.col("a").fill_null(0) + pl.col("b").fill_null(0)).alias("a+b")
)
print(total_fillna)

# Using fill_null as a "fill_value" for multiplication
print("\nUsing fill_value (1) for multiplication:")
product_fillna = joined.with_columns(
    (pl.col("a").fill_null(1) * pl.col("b").fill_null(1)).alias("a*b")
)
print(product_fillna)


Adding by setting null values to 0:
shape: (4, 4)
┌───────┬──────┬──────┬─────┐
│ label ┆ a    ┆ b    ┆ a+b │
│ ---   ┆ ---  ┆ ---  ┆ --- │
│ str   ┆ i64  ┆ i64  ┆ i64 │
╞═══════╪══════╪══════╪═════╡
│ a     ┆ 10   ┆ null ┆ 10  │
│ b     ┆ 20   ┆ 5    ┆ 25  │
│ c     ┆ 30   ┆ 15   ┆ 45  │
│ d     ┆ null ┆ 25   ┆ 25  │
└───────┴──────┴──────┴─────┘

Using fill_value (1) for multiplication:
shape: (4, 4)
┌───────┬──────┬──────┬─────┐
│ label ┆ a    ┆ b    ┆ a*b │
│ ---   ┆ ---  ┆ ---  ┆ --- │
│ str   ┆ i64  ┆ i64  ┆ i64 │
╞═══════╪══════╪══════╪═════╡
│ a     ┆ 10   ┆ null ┆ 10  │
│ b     ┆ 20   ┆ 5    ┆ 100 │
│ c     ┆ 30   ┆ 15   ┆ 450 │
│ d     ┆ null ┆ 25   ┆ 25  │
└───────┴──────┴──────┴─────┘


In [33]:
# Cumulative operations
series = pl.Series("s", [2, 3, 1, 4, 5])
print("Original series:", series.to_numpy())
print("Cumulative sum:", series.cum_sum().to_numpy())
print("Cumulative product:", series.cum_prod().to_numpy())
print("Cumulative maximum:", series.cum_max().to_numpy())
print("Cumulative minimum:", series.cum_min().to_numpy())

Original series: [2 3 1 4 5]
Cumulative sum: [ 2  5  6 10 15]
Cumulative product: [  2   6   6  24 120]
Cumulative maximum: [2 3 3 4 5]
Cumulative minimum: [2 2 1 1 1]


In [34]:
# Basic statistics
data_series = pl.Series("data", [23, 45, 12, 67, 34, 89, 15, 42])

print("Series:", data_series.to_numpy())
print("Mean:", data_series.mean())
print("Median:", data_series.median())
print("Standard Deviation:", data_series.std())  # sample std (ddof=1), same as Pandas
print("Variance:", data_series.var())  # sample variance (ddof=1), same as Pandas
print("Minimum:", data_series.min())
print("Maximum:", data_series.max())
print("Sum:", data_series.sum())

# Polars' default quantile interpolation is "nearest"; Pandas uses "linear"
print("1st Quartile (Q1) (linear):", data_series.quantile(0.25, interpolation="linear"))
print("1st Quartile (Q1) (nearest):", data_series.quantile(0.25))

print("3rd Quartile (Q3) (linear):", data_series.quantile(0.75, interpolation="linear"))
print("3rd Quartile (Q3) (nearest):", data_series.quantile(0.75))

Series: [23 45 12 67 34 89 15 42]
Mean: 40.875
Median: 38.0
Standard Deviation: 26.4220551812307
Variance: 698.125
Minimum: 12
Maximum: 89
Sum: 327
1st Quartile (Q1) (linear): 21.0
1st Quartile (Q1) (nearest): 23.0
3rd Quartile (Q3) (linear): 50.5
3rd Quartile (Q3) (nearest): 45.0


In [35]:
grades = pl.DataFrame(
    {"name": ["John", "Jim", "Jack", "Jill"], "grade": [45, 70, 85, 30]}
)

# Add 10 points to those below 50
# Polars is immutable: we create a new DataFrame with when/then/otherwise
grades = grades.with_columns(
    pl.when(pl.col("grade") < 50)
    .then(pl.col("grade") + 10)
    .otherwise(pl.col("grade"))
    .alias("grade")
)
print(grades)

shape: (4, 2)
┌──────┬───────┐
│ name ┆ grade │
│ ---  ┆ ---   │
│ str  ┆ i64   │
╞══════╪═══════╡
│ John ┆ 55    │
│ Jim  ┆ 70    │
│ Jack ┆ 85    │
│ Jill ┆ 40    │
└──────┴───────┘


In [36]:
# Comparison operations
series = pl.Series("s", [15, 25, 35, 45, 55])

print("Series > 30:", (series > 30).to_list())
print("Series == 35:", (series == 35).to_list())
print("Series <= 40:", (series <= 40).to_list())

# Filtering with boolean masks
print("Values greater than 30:", series.filter(series > 30).to_list())
print(
    "Values between 20 and 50:",
    series.filter(series.is_between(20, 50, closed="none")).to_list(),
)

Series > 30: [False, False, True, True, True]
Series == 35: [False, False, True, False, False]
Series <= 40: [True, True, True, False, False]
Values greater than 30: [35, 45, 55]
Values between 20 and 50: [25, 35, 45]


In [37]:
# Applying custom function
def custom_function(x):
    return x**2 + 2 * x + 1


series = pl.Series("s", [1, 2, 3, 4, 5])

# 1) Preferred: the function only uses arithmetic, so it already works vectorized on a Series
result = custom_function(series)
print("Custom function result (vectorized):", result.to_list())

# 2) map_elements calls Python once per element: flexible but slow. Use it only when
#    there is no expression equivalent. (Polars deliberately prints a
#    PolarsInefficientMapWarning below and suggests the native expression.)
result = series.map_elements(custom_function, return_dtype=pl.Int64)
print("Custom function result (map_elements):", result.to_list())

# Lambda functions (same idea)
square = series.map_elements(lambda x: x**2, return_dtype=pl.Int64)
print("Squaring with lambda:", square.to_list())
# ... but the vectorized form is faster and clearer:
print("Squaring vectorized:", (series**2).to_list())

Custom function result (vectorized): [4, 9, 16, 25, 36]
Custom function result (map_elements): [4, 9, 16, 25, 36]
Squaring with lambda: [1, 4, 9, 16, 25]
Squaring vectorized: [1, 4, 9, 16, 25]


/var/folders/nc/kj2v9lkd7jx7tty3mdq2f2040000gn/T/ipykernel_10661/3682919242.py:15: PolarsInefficientMapWarning: 
Series.map_elements is significantly slower than the native series API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - s.map_elements(custom_function)
with this one instead:
  + ((s ** 2) + (2 * s)) + 1

  result = series.map_elements(custom_function, return_dtype=pl.Int64)
/var/folders/nc/kj2v9lkd7jx7tty3mdq2f2040000gn/T/ipykernel_10661/3682919242.py:19: PolarsInefficientMapWarning: 
Series.map_elements is significantly slower than the native series API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - s.map_elements(lambda x: ...)
with this one instead:
  + s ** 2

  square = series.map_elements(lambda x: x**2, return_dtype=pl.Int64)


In [38]:
print("\n" + "=" * 50)
print("REAL LIFE EXAMPLE: Sales Analysis")
print("=" * 50)

# Sales data (the month labels live in a column, there is no index)
monthly = pl.DataFrame(
    {
        "month": ["January", "February", "March", "April", "May"],
        "sales": [15000, 22000, 18000, 25000, 30000],
        "expenses": [8000, 9500, 12000, 11000, 14000],
    }
)

print("Monthly Sales:")
print(monthly.select("month", "sales"))

print("\nMonthly Expenses:")
print(monthly.select("month", "expenses"))

# Profit, percentage change and cumulative sales in one with_columns call
monthly = monthly.with_columns(
    (pl.col("sales") - pl.col("expenses")).alias("profit"),
    (pl.col("sales").pct_change() * 100).round(2).alias("sales_pct_change"),
    pl.col("sales").cum_sum().alias("cumulative_sales"),
)

print("\nMonthly Profit:")
print(monthly.select("month", "profit"))

print("\nMonthly % Change in Sales:")
print(monthly.select("month", "sales_pct_change"))

print("\nCumulative Sales:")
print(monthly.select("month", "cumulative_sales"))

# Performance above average
print("\nMonths with Above Average Sales:")
print(monthly.filter(pl.col("sales") > pl.col("sales").mean()).select("month", "sales"))


REAL LIFE EXAMPLE: Sales Analysis
Monthly Sales:
shape: (5, 2)
┌──────────┬───────┐
│ month    ┆ sales │
│ ---      ┆ ---   │
│ str      ┆ i64   │
╞══════════╪═══════╡
│ January  ┆ 15000 │
│ February ┆ 22000 │
│ March    ┆ 18000 │
│ April    ┆ 25000 │
│ May      ┆ 30000 │
└──────────┴───────┘

Monthly Expenses:
shape: (5, 2)
┌──────────┬──────────┐
│ month    ┆ expenses │
│ ---      ┆ ---      │
│ str      ┆ i64      │
╞══════════╪══════════╡
│ January  ┆ 8000     │
│ February ┆ 9500     │
│ March    ┆ 12000    │
│ April    ┆ 11000    │
│ May      ┆ 14000    │
└──────────┴──────────┘

Monthly Profit:
shape: (5, 2)
┌──────────┬────────┐
│ month    ┆ profit │
│ ---      ┆ ---    │
│ str      ┆ i64    │
╞══════════╪════════╡
│ January  ┆ 7000   │
│ February ┆ 12500  │
│ March    ┆ 6000   │
│ April    ┆ 14000  │
│ May      ┆ 16000  │
└──────────┴────────┘

Monthly % Change in Sales:
shape: (5, 2)
┌──────────┬──────────────────┐
│ month    ┆ sales_pct_change │
│ ---      ┆ ---             

In [39]:
# Method chaining example
result = (
    pl.DataFrame({"v": [10, 25, 35, 42, 58]})
    .with_columns((pl.col("v") * 1.1).round(1))  # 10% increase + Round
    .filter(pl.col("v") > 30)  # Take those greater than 30 (no NaN cleanup needed)
)

print("Method chaining result:")
print(result)

# The same query in lazy mode: nothing runs until collect()
lazy_result = (
    pl.LazyFrame({"v": [10, 25, 35, 42, 58]})
    .with_columns((pl.col("v") * 1.1).round(1))
    .filter(pl.col("v") > 30)
)
print("\nLazy query plan:")
print(lazy_result.explain())
print(lazy_result.collect())

Method chaining result:
shape: (3, 1)
┌──────┐
│ v    │
│ ---  │
│ f64  │
╞══════╡
│ 38.5 │
│ 46.2 │
│ 63.8 │
└──────┘

Lazy query plan:
FILTER col("v") > 30.0
FROM
   WITH_COLUMNS:
   [((col("v").cast(Float64) * 1.1)).round()] 
    DF ["v"]; PROJECT */1 COLUMNS
shape: (3, 1)
┌──────┐
│ v    │
│ ---  │
│ f64  │
╞══════╡
│ 38.5 │
│ 46.2 │
│ 63.8 │
└──────┘


In [40]:
# Broadcast operations
series = pl.Series("s", [1, 2, 3, 4, 5])

# Apply same operation to all elements
print("Series + 10:", (series + 10).to_list())
print("Series * 2:", (series * 2).to_list())

# Broadcast with other series: only a length-1 Series is broadcast
other_series = pl.Series("o", [100])  # Single element series
print("Series + single element series:", (series + other_series).to_list())

Series + 10: [11, 12, 13, 14, 15]
Series * 2: [2, 4, 6, 8, 10]
Series + single element series: [101, 102, 103, 104, 105]


**Important Notes:**

* Vectorization: Polars series support vectorized operations like numpy

* No Index Alignment: Operations are positional; lengths must match (or one side must have length 1). To align on labels, `join` first

* Missing values: Missing data is `null` (for every dtype). `NaN` is a separate float value. Use `fill_null` / `drop_nulls`; use `fill_nan` for NaN

* Immutability: Polars objects are not modified in place. `series += 10` creates a new Series; for DataFrames use `with_columns` and reassign

* Expressions over `map_elements`: Built-in expressions run in parallel in Rust, Python callbacks run row by row

* Performance: Vectorized operations are much faster than loops

These operations make polars series a powerful and flexible data processing tool!

# **Polars DataFrame**

DataFrame is the most important data structure of the Polars library. It is a 2-dimensional, typed data structure like an Excel table.

DataFrames are structures formed by the combination of equal-length Series. They are often likened to SQL tables with columns and rows of different types. DataFrames allow us to process data more easily. They consist of named Columns and rows; rows are identified by position only (there is **no index**). Data analysis can be performed on these values with expressions.

Two important differences from Pandas:

* **No index**: if you need a row label, keep it in a normal column.
* **Immutable style**: methods such as `with_columns`, `filter`, `sort` return a new DataFrame, so assign the result (`df = df.with_columns(...)`) if you want to keep it.

# Creating Polars DataFrame

In [41]:
print("=" * 50)
print("DATAFRAME CREATION METHODS")
print("=" * 50)

DATAFRAME CREATION METHODS


# Creating DataFrame with Dictionary (most common method)

In [42]:
# Most common method
data = {
    "Name": ["Jack", "Jim", "John", "Jill", "Jenny"],
    "Age": [25, 30, 35, 28, 32],
    "City": ["New York", "Boston", "Los Angles", "Miami", "Chicago"],
    "Salary": [50000, 60000, 70000, 55000, 65000],
}

df = pl.DataFrame(data)
print("DataFrame created with dictionary:")
print(df)
print(f"Size: {df.shape}")  # (5, 4)
print(f"Shape: {df.height} rows, {df.width} columns")

DataFrame created with dictionary:
shape: (5, 4)
┌───────┬─────┬────────────┬────────┐
│ Name  ┆ Age ┆ City       ┆ Salary │
│ ---   ┆ --- ┆ ---        ┆ ---    │
│ str   ┆ i64 ┆ str        ┆ i64    │
╞═══════╪═════╪════════════╪════════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  │
│ Jill  ┆ 28  ┆ Miami      ┆ 55000  │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  │
└───────┴─────┴────────────┴────────┘
Size: (5, 4)
Shape: 5 rows, 4 columns


# Creating DataFrame with Lists

In [43]:
# List of lists
values = [
    ["Jack", 25, "New York", 50000],
    ["Jim", 30, "Boston", 60000],
    ["John", 35, "Los Angles", 70000],
    ["Jill", 28, "Miami", 55000],
    ["Jenny", 32, "Chicago", 65000],
]

columns = ["Name", "Age", "City", "Salary"]
# orient="row": each inner list is a row
df_list = pl.DataFrame(values, schema=columns, orient="row")
print("\nDataFrame created with list:")
print(df_list)


DataFrame created with list:
shape: (5, 4)
┌───────┬─────┬────────────┬────────┐
│ Name  ┆ Age ┆ City       ┆ Salary │
│ ---   ┆ --- ┆ ---        ┆ ---    │
│ str   ┆ i64 ┆ str        ┆ i64    │
╞═══════╪═════╪════════════╪════════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  │
│ Jill  ┆ 28  ┆ Miami      ┆ 55000  │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  │
└───────┴─────┴────────────┴────────┘


# Creating with Numpy Array

In [44]:
# From Numpy array
np_array = np.array(
    [[1, "Jack", 85.5], [2, "Jim", 92.0], [3, "John", 78.5], [4, "Jill", 88.0]]
)

# A NumPy array has a single dtype, so mixed content is coerced to strings
df_numpy = pl.DataFrame(np_array, schema=["ID", "Name", "Grade"], orient="row")
print("\nDataFrame created with Numpy array:")
print(df_numpy)

# Convert to the proper types explicitly (Polars is strictly typed, no silent coercion)
df_numpy = df_numpy.cast({"ID": pl.Int64, "Grade": pl.Float64})
print(df_numpy.schema)


DataFrame created with Numpy array:
shape: (4, 3)
┌─────┬──────┬───────┐
│ ID  ┆ Name ┆ Grade │
│ --- ┆ ---  ┆ ---   │
│ str ┆ str  ┆ str   │
╞═════╪══════╪═══════╡
│ 1   ┆ Jack ┆ 85.5  │
│ 2   ┆ Jim  ┆ 92.0  │
│ 3   ┆ John ┆ 78.5  │
│ 4   ┆ Jill ┆ 88.0  │
└─────┴──────┴───────┘
Schema({'ID': Int64, 'Name': String, 'Grade': Float64})


# Accessing General DataFrame Properties

In [45]:
print("\n" + "=" * 50)
print("BASIC DATAFRAME PROPERTIES")
print("=" * 50)

data = {
    "Name": ["Jack", "Jim", "John", "Jill", "Jenny"],
    "Age": [25, 30, 35, 28, 32],
    "City": ["New York", "Boston", "Los Angles", "Miami", "Chicago"],
    "Salary": [50000, 60000, 70000, 55000, 65000],
}

df = pl.DataFrame(data)

print("DataFrame info:")
df.glimpse()  # Polars' counterpart of df.info() (prints directly)

print(f"\nSize: {df.shape}")
print(f"Row count: {len(df)}")
print(f"Column count: {len(df.columns)}")
print(f"Column names: {df.columns}")
print("Index info: Polars has no index (rows are addressed by position)")
print(f"Data types:\n{df.schema}")

print("\nFirst 3 rows:")
print(df.head(3))

print("\nLast 2 rows:")
print(df.tail(2))


BASIC DATAFRAME PROPERTIES
DataFrame info:
Rows: 5
Columns: 4
$ Name   <str> 'Jack', 'Jim', 'John', 'Jill', 'Jenny'
$ Age    <i64> 25, 30, 35, 28, 32
$ City   <str> 'New York', 'Boston', 'Los Angles', 'Miami', 'Chicago'
$ Salary <i64> 50000, 60000, 70000, 55000, 65000


Size: (5, 4)
Row count: 5
Column count: 4
Column names: ['Name', 'Age', 'City', 'Salary']
Index info: Polars has no index (rows are addressed by position)
Data types:
Schema({'Name': String, 'Age': Int64, 'City': String, 'Salary': Int64})

First 3 rows:
shape: (3, 4)
┌──────┬─────┬────────────┬────────┐
│ Name ┆ Age ┆ City       ┆ Salary │
│ ---  ┆ --- ┆ ---        ┆ ---    │
│ str  ┆ i64 ┆ str        ┆ i64    │
╞══════╪═════╪════════════╪════════╡
│ Jack ┆ 25  ┆ New York   ┆ 50000  │
│ Jim  ┆ 30  ┆ Boston     ┆ 60000  │
│ John ┆ 35  ┆ Los Angles ┆ 70000  │
└──────┴─────┴────────────┴────────┘

Last 2 rows:
shape: (2, 4)
┌───────┬─────┬─────────┬────────┐
│ Name  ┆ Age ┆ City    ┆ Salary │
│ ---   ┆ --- ┆ ---     ┆ ---

# Column Operations

In [46]:
print("\n" + "=" * 50)
print("COLUMN OPERATIONS")
print("=" * 50)

# Selecting column
print("Name column:")
print(df["Name"])
print(type(df["Name"]))  # polars Series

# Selecting multiple columns
print("\nName and Age columns:")
print(df.select("Name", "Age"))

# Adding new column (with_columns returns a NEW DataFrame, so we reassign)
df = df.with_columns(
    pl.Series("Experience", [3, 5, 8, 4, 6]),
    (pl.col("Salary") * 12).alias("Annual Salary"),
    pl.when(pl.col("Age") < 30)
    .then(pl.lit("Young"))
    .otherwise(pl.lit("Adult"))
    .alias("Age Group"),  # np.where equivalent
)

print("\nNew columns added:")
print(df)

# Deleting column
df_drop = df.drop("Age Group")  # no axis argument needed
print("\nAge Group column dropped:")
print(df_drop.columns)


COLUMN OPERATIONS
Name column:
shape: (5,)
Series: 'Name' [str]
[
	"Jack"
	"Jim"
	"John"
	"Jill"
	"Jenny"
]
<class 'polars.series.series.Series'>

Name and Age columns:
shape: (5, 2)
┌───────┬─────┐
│ Name  ┆ Age │
│ ---   ┆ --- │
│ str   ┆ i64 │
╞═══════╪═════╡
│ Jack  ┆ 25  │
│ Jim   ┆ 30  │
│ John  ┆ 35  │
│ Jill  ┆ 28  │
│ Jenny ┆ 32  │
└───────┴─────┘

New columns added:
shape: (5, 7)
┌───────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       │
│ str   ┆ i64 ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       │
╞═══════╪═════╪════════════╪════════╪════════════╪═══════════════╪═══════════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  ┆ 3          ┆ 600000        ┆ Young     │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  ┆ 5          ┆ 720000        ┆ Adult     │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  ┆ 8          ┆ 84

# Row Operations

In [48]:
print("\n" + "=" * 50)
print("ROW OPERATIONS")
print("=" * 50)

# Selecting row by position (there is no .loc/.iloc)
print("First row (as a tuple / as a dict):")
print(df.row(0))
print(df.row(0, named=True))

print("\nFirst 3 rows:")
print(df[:3])  # same as df.head(3) / df.slice(0, 3)

print("\nRow at specific position (3rd row):")
print(df[2])  # a one-row DataFrame

# Selecting rows by condition
print("\nThose older than 30:")
print(df.filter(pl.col("Age") > 30))

print("\nThose living in Istanbul:")
print(df.filter(pl.col("City") == "Istanbul"))

# Multiple condition (several predicates are AND-ed)
print("\nThose older than 30 AND salary more than 60000:")
print(df.filter(pl.col("Age") > 30, pl.col("Salary") > 60000))


ROW OPERATIONS
First row (as a tuple / as a dict):
('Jack', 25, 'New York', 50000, 3, 600000, 'Young')
{'Name': 'Jack', 'Age': 25, 'City': 'New York', 'Salary': 50000, 'Experience': 3, 'Annual Salary': 600000, 'Age Group': 'Young'}

First 3 rows:
shape: (3, 7)
┌──────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┐
│ Name ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group │
│ ---  ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       │
│ str  ┆ i64 ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       │
╞══════╪═════╪════════════╪════════╪════════════╪═══════════════╪═══════════╡
│ Jack ┆ 25  ┆ New York   ┆ 50000  ┆ 3          ┆ 600000        ┆ Young     │
│ Jim  ┆ 30  ┆ Boston     ┆ 60000  ┆ 5          ┆ 720000        ┆ Adult     │
│ John ┆ 35  ┆ Los Angles ┆ 70000  ┆ 8          ┆ 840000        ┆ Adult     │
└──────┴─────┴────────────┴────────┴────────────┴───────────────┴───────────┘

Row at specific position (3rd row):

# Row Labels Instead of an Index

Polars has no index. The common index tasks map to:

* an auto-incrementing id → `with_row_index`
* a label/key → a normal column (use `filter`, `join` or `partition_by` to look it up)

In [49]:
print("\n" + "=" * 50)
print("INDEX OPERATIONS (no index in Polars)")
print("=" * 50)

# "Setting custom index": add the labels as a column
df_custom = df.with_columns(pl.Series("label", ["A", "B", "C", "D", "E"]))
print("DataFrame with a label column:")
print(df_custom)

# "Resetting index": a 0..n-1 counter column (UInt32)
df_reset = df_custom.drop("label").with_row_index()
print("\nRow counter added:")
print(df_reset)

# "Making index a column": rename the counter to the name you want
df_custom = df_custom.drop("label").with_row_index("ID")
print("\nCounter as ID column:")
print(df_custom)

# "Making a column the index": no need, but for fast key lookups you can split by key
by_name = df.partition_by("Name", as_dict=True, include_key=True)
print("\nLookup by Name (dict of DataFrames):")
print(by_name[("John",)])


INDEX OPERATIONS (no index in Polars)
DataFrame with a label column:
shape: (5, 8)
┌───────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┬───────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group ┆ label │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       ┆ ---   │
│ str   ┆ i64 ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       ┆ str   │
╞═══════╪═════╪════════════╪════════╪════════════╪═══════════════╪═══════════╪═══════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  ┆ 3          ┆ 600000        ┆ Young     ┆ A     │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  ┆ 5          ┆ 720000        ┆ Adult     ┆ B     │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  ┆ 8          ┆ 840000        ┆ Adult     ┆ C     │
│ Jill  ┆ 28  ┆ Miami      ┆ 55000  ┆ 4          ┆ 660000        ┆ Young     ┆ D     │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  ┆ 6          ┆ 780000        ┆ Adult     ┆ E     │
└───────┴─────┴────────────┴────────┴─────────

# Data Filtering and Querying

In [50]:
print("\n" + "=" * 50)
print("DATA FILTERING AND QUERYING")
print("=" * 50)

# Pandas' .query() → filter() with expressions (or SQL)
print("Filtering with expressions:")
print(df.filter((pl.col("Age") > 30) & (pl.col("Salary") > 60000)))

# SQL interface (the DataFrame is available as the table "self")
print("\nQuerying with SQL:")
print(df.sql("SELECT * FROM self WHERE Age > 30 AND Salary > 60000"))

# Filtering with .is_in()
cities = ["Boston", "New York"]
print(f"\nThose living in {cities} cities:")
print(df.filter(pl.col("City").is_in(cities)))

# Filtering with string operations
print("\nNames starting with 'A':")
print(df.filter(pl.col("Name").str.starts_with("A")))

print("\nNames starting with 'J':")
print(df.filter(pl.col("Name").str.starts_with("J")))

# Range querying with is_between (inclusive on both ends by default)
print("\nThose aged between 25-35:")
print(df.filter(pl.col("Age").is_between(25, 35)))


DATA FILTERING AND QUERYING
Filtering with expressions:
shape: (2, 7)
┌───────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       │
│ str   ┆ i64 ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       │
╞═══════╪═════╪════════════╪════════╪════════════╪═══════════════╪═══════════╡
│ John  ┆ 35  ┆ Los Angles ┆ 70000  ┆ 8          ┆ 840000        ┆ Adult     │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  ┆ 6          ┆ 780000        ┆ Adult     │
└───────┴─────┴────────────┴────────┴────────────┴───────────────┴───────────┘

Querying with SQL:
shape: (2, 7)
┌───────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       │
│ str   ┆ i64 ┆ str       

# Data Grouping and Aggregation

In [51]:
print("\n" + "=" * 50)
print("DATA GROUPING AND AGGREGATION")
print("=" * 50)

# Grouping operations
# (group_by does not guarantee row order, so we sort for a stable output)
print("Average salary by city:")
print(df.group_by("City").agg(pl.col("Salary").mean()).sort("City"))

print("\nDetailed statistics by city:")
print(
    df.group_by("City")
    .agg(
        pl.col("Salary").mean().alias("Salary_mean"),
        pl.col("Salary").min().alias("Salary_min"),
        pl.col("Salary").max().alias("Salary_max"),
        pl.col("Salary").std().alias("Salary_std"),
        pl.col("Age").mean().alias("Age_mean"),
        pl.col("Age").count().alias("Age_count"),
    )
    .sort("City")
)

# Multiple grouping
df = df.with_columns(pl.Series("Gender", ["M", "M", "M", "F", "F"]))
print("\nGrouping by City and Gender:")
print(df.group_by("City", "Gender").agg(pl.col("Salary").mean()).sort("City", "Gender"))


DATA GROUPING AND AGGREGATION
Average salary by city:
shape: (5, 2)
┌────────────┬─────────┐
│ City       ┆ Salary  │
│ ---        ┆ ---     │
│ str        ┆ f64     │
╞════════════╪═════════╡
│ Boston     ┆ 60000.0 │
│ Chicago    ┆ 65000.0 │
│ Los Angles ┆ 70000.0 │
│ Miami      ┆ 55000.0 │
│ New York   ┆ 50000.0 │
└────────────┴─────────┘

Detailed statistics by city:
shape: (5, 7)
┌────────────┬─────────────┬────────────┬────────────┬────────────┬──────────┬───────────┐
│ City       ┆ Salary_mean ┆ Salary_min ┆ Salary_max ┆ Salary_std ┆ Age_mean ┆ Age_count │
│ ---        ┆ ---         ┆ ---        ┆ ---        ┆ ---        ┆ ---      ┆ ---       │
│ str        ┆ f64         ┆ i64        ┆ i64        ┆ f64        ┆ f64      ┆ u32       │
╞════════════╪═════════════╪════════════╪════════════╪════════════╪══════════╪═══════════╡
│ Boston     ┆ 60000.0     ┆ 60000      ┆ 60000      ┆ null       ┆ 30.0     ┆ 1         │
│ Chicago    ┆ 65000.0     ┆ 65000      ┆ 65000      ┆ null       

# Data Sorting

In [52]:
print("\n" + "=" * 50)
print("DATA SORTING")
print("=" * 50)

# Sorting by single column
print("Ascending sort by Salary:")
print(df.sort("Salary"))

print("\nDescending sort by Salary:")
print(df.sort("Salary", descending=True))

# Sorting by multiple columns
print("\nSort by City first, then Salary:")
print(df.sort(["City", "Salary"], descending=[False, True]))


DATA SORTING
Ascending sort by Salary:
shape: (5, 8)
┌───────┬─────┬────────────┬────────┬────────────┬───────────────┬───────────┬────────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group ┆ Gender │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       ┆ ---    │
│ str   ┆ i64 ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       ┆ str    │
╞═══════╪═════╪════════════╪════════╪════════════╪═══════════════╪═══════════╪════════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  ┆ 3          ┆ 600000        ┆ Young     ┆ M      │
│ Jill  ┆ 28  ┆ Miami      ┆ 55000  ┆ 4          ┆ 660000        ┆ Young     ┆ F      │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  ┆ 5          ┆ 720000        ┆ Adult     ┆ M      │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  ┆ 6          ┆ 780000        ┆ Adult     ┆ F      │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  ┆ 8          ┆ 840000        ┆ Adult     ┆ M      │
└───────┴─────┴────────────┴────────┴────────────┴───────────────┴

# Missing Data Operations

In Polars missing data is `null` for every dtype. `NaN` is a regular floating point value and is **not** treated as missing by `is_null()` / `drop_nulls()`.

In [53]:
print("\n" + "=" * 50)
print("MISSING DATA OPERATIONS")
print("=" * 50)

# Creating DataFrame with missing data (set the value at a given row position to null;
# pl.int_range(pl.len()) is the row position)
df_missing = df.with_columns(
    pl.when(pl.int_range(pl.len()) == 0)
    .then(None)
    .otherwise(pl.col("Age"))
    .alias("Age"),
    pl.when(pl.int_range(pl.len()) == 2)
    .then(None)
    .otherwise(pl.col("Salary"))
    .alias("Salary"),
    pl.when(pl.int_range(pl.len()) == 4)
    .then(None)
    .otherwise(pl.col("City"))
    .alias("City"),
)

print("DataFrame with missing data:")
print(df_missing)

print("\nMissing data check:")
print(df_missing.select(pl.all().is_null()))

print("\nMissing data counts:")
print(df_missing.null_count())

# Cleaning missing data
df_clean = df_missing.drop_nulls()  # Drop rows with missing data
print("\nMissing data dropped:")
print(df_clean)

# Filling missing data
df_filled = df_missing.with_columns(
    pl.col("Age").fill_null(pl.col("Age").mean()),
    pl.col("Salary").fill_null(pl.col("Salary").median()),
    pl.col("City").fill_null("Unknown"),
)
print("\nMissing data filled:")
print(df_filled)

# null vs NaN
s = pl.Series("x", [1.0, None, float("nan")])
print("\nnull vs NaN:")
print("is_null:", s.is_null().to_list())  # [False, True, False]
print("is_nan: ", s.is_nan().to_list())  # [False, None, True]
print("fill_nan(None) converts NaN to null:", s.fill_nan(None).is_null().to_list())


MISSING DATA OPERATIONS
DataFrame with missing data:
shape: (5, 8)
┌───────┬──────┬────────────┬────────┬────────────┬───────────────┬───────────┬────────┐
│ Name  ┆ Age  ┆ City       ┆ Salary ┆ Experience ┆ Annual Salary ┆ Age Group ┆ Gender │
│ ---   ┆ ---  ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       ┆ ---    │
│ str   ┆ i64  ┆ str        ┆ i64    ┆ i64        ┆ i64           ┆ str       ┆ str    │
╞═══════╪══════╪════════════╪════════╪════════════╪═══════════════╪═══════════╪════════╡
│ Jack  ┆ null ┆ New York   ┆ 50000  ┆ 3          ┆ 600000        ┆ Young     ┆ M      │
│ Jim   ┆ 30   ┆ Boston     ┆ 60000  ┆ 5          ┆ 720000        ┆ Adult     ┆ M      │
│ John  ┆ 35   ┆ Los Angles ┆ null   ┆ 8          ┆ 840000        ┆ Adult     ┆ M      │
│ Jill  ┆ 28   ┆ Miami      ┆ 55000  ┆ 4          ┆ 660000        ┆ Young     ┆ F      │
│ Jenny ┆ 32   ┆ null       ┆ 65000  ┆ 6          ┆ 780000        ┆ Adult     ┆ F      │
└───────┴──────┴────────────┴────────┴────

# Pivot Tables

In [54]:
print("\n" + "=" * 50)
print("PIVOT TABLES")
print("=" * 50)

# Let's add more data
more_data = {
    "Region": ["North", "South", "West", "East", "North", "South"],
    "Product": ["A", "B", "A", "B", "A", "B"],
    "Sales": [100, 150, 200, 120, 180, 160],
    "Quarter": [1, 1, 2, 2, 3, 3],
}
df_sales = pl.DataFrame(more_data)

print("Sales data:")
print(df_sales)

# Creating pivot table
pivot = (
    df_sales.pivot(
        on="Quarter", index="Region", values="Sales", aggregate_function="sum"
    )
    .fill_null(0)
    .sort("Region")
)
print("\nPivot Table:")
print(pivot)


PIVOT TABLES
Sales data:
shape: (6, 4)
┌────────┬─────────┬───────┬─────────┐
│ Region ┆ Product ┆ Sales ┆ Quarter │
│ ---    ┆ ---     ┆ ---   ┆ ---     │
│ str    ┆ str     ┆ i64   ┆ i64     │
╞════════╪═════════╪═══════╪═════════╡
│ North  ┆ A       ┆ 100   ┆ 1       │
│ South  ┆ B       ┆ 150   ┆ 1       │
│ West   ┆ A       ┆ 200   ┆ 2       │
│ East   ┆ B       ┆ 120   ┆ 2       │
│ North  ┆ A       ┆ 180   ┆ 3       │
│ South  ┆ B       ┆ 160   ┆ 3       │
└────────┴─────────┴───────┴─────────┘

Pivot Table:
shape: (4, 4)
┌────────┬─────┬─────┬─────┐
│ Region ┆ 1   ┆ 2   ┆ 3   │
│ ---    ┆ --- ┆ --- ┆ --- │
│ str    ┆ i64 ┆ i64 ┆ i64 │
╞════════╪═════╪═════╪═════╡
│ East   ┆ 0   ┆ 120 ┆ 0   │
│ North  ┆ 100 ┆ 0   ┆ 180 │
│ South  ┆ 150 ┆ 0   ┆ 160 │
│ West   ┆ 0   ┆ 200 ┆ 0   │
└────────┴─────┴─────┴─────┘


# DataFrame Merging

In [55]:
print("\n" + "=" * 50)
print("DATAFRAME MERGING OPERATIONS")
print("=" * 50)

# Let's create two DataFrames
df1 = pl.DataFrame(
    {
        "ID": [1, 2, 3, 4],
        "Name": ["John", "Jill", "Jack", "Jenny"],
        "Department": ["IT", "HR", "IT", "Finance"],
    }
)

df2 = pl.DataFrame(
    {
        "ID": [1, 2, 3, 5],
        "Salary": [50000, 60000, 70000, 80000],
        "Experience": [3, 5, 8, 10],
    }
)

print("DF1:")
print(df1)
print("\nDF2:")
print(df2)

# Inner join
inner_merge = df1.join(df2, on="ID", how="inner")
print("\nInner Join:")
print(inner_merge)

# Left join
left_merge = df1.join(df2, on="ID", how="left")
print("\nLeft Join:")
print(left_merge)

# Full (outer) join: coalesce=True merges the two key columns into one
full_merge = df1.join(df2, on="ID", how="full", coalesce=True).sort("ID")
print("\nFull Join:")
print(full_merge)

# Concatenation
df3 = pl.DataFrame(
    {"ID": [5, 6], "Name": ["Jimmy", "Jony"], "Department": ["IT", "HR"]}
)

merged = pl.concat([df1, df3])  # no ignore_index needed: there is no index
print("\nMerging with Concat:")
print(merged)


DATAFRAME MERGING OPERATIONS
DF1:
shape: (4, 3)
┌─────┬───────┬────────────┐
│ ID  ┆ Name  ┆ Department │
│ --- ┆ ---   ┆ ---        │
│ i64 ┆ str   ┆ str        │
╞═════╪═══════╪════════════╡
│ 1   ┆ John  ┆ IT         │
│ 2   ┆ Jill  ┆ HR         │
│ 3   ┆ Jack  ┆ IT         │
│ 4   ┆ Jenny ┆ Finance    │
└─────┴───────┴────────────┘

DF2:
shape: (4, 3)
┌─────┬────────┬────────────┐
│ ID  ┆ Salary ┆ Experience │
│ --- ┆ ---    ┆ ---        │
│ i64 ┆ i64    ┆ i64        │
╞═════╪════════╪════════════╡
│ 1   ┆ 50000  ┆ 3          │
│ 2   ┆ 60000  ┆ 5          │
│ 3   ┆ 70000  ┆ 8          │
│ 5   ┆ 80000  ┆ 10         │
└─────┴────────┴────────────┘

Inner Join:
shape: (3, 5)
┌─────┬──────┬────────────┬────────┬────────────┐
│ ID  ┆ Name ┆ Department ┆ Salary ┆ Experience │
│ --- ┆ ---  ┆ ---        ┆ ---    ┆ ---        │
│ i64 ┆ str  ┆ str        ┆ i64    ┆ i64        │
╞═════╪══════╪════════════╪════════╪════════════╡
│ 1   ┆ John ┆ IT         ┆ 50000  ┆ 3          │
│ 2   ┆ Jill ┆

# Data Types and Conversions

In [56]:
print("\n" + "=" * 50)
print("DATA TYPES AND CONVERSIONS")
print("=" * 50)

print("Data types:")
print(df.schema)

# Data type conversion (cast is strict by default and raises on failure)
df_type = df.with_columns(
    pl.col("Age").cast(pl.Float64),
    pl.col("Name").cast(pl.String),
)

print("\nAfter type conversion:")
print(df_type.schema)

# Categorical data
df = df.with_columns(pl.col("City").cast(pl.Categorical))
print("\nCategorical data:")
print(df["City"].dtype)
print(df["City"].unique().sort())  # distinct categories present in the column

# For a fixed, ordered list of categories use pl.Enum (the dtype carries its categories)
city_enum = pl.Enum(["Boston", "Chicago", "Los Angles", "Miami", "New York"])
print(df["City"].cast(pl.String).cast(city_enum).dtype.categories.to_list())


DATA TYPES AND CONVERSIONS
Data types:
Schema({'Name': String, 'Age': Int64, 'City': String, 'Salary': Int64, 'Experience': Int64, 'Annual Salary': Int64, 'Age Group': String, 'Gender': String})

After type conversion:
Schema({'Name': String, 'Age': Float64, 'City': String, 'Salary': Int64, 'Experience': Int64, 'Annual Salary': Int64, 'Age Group': String, 'Gender': String})

Categorical data:
Categorical
shape: (5,)
Series: 'City' [cat]
[
	"Boston"
	"Chicago"
	"Los Angles"
	"Miami"
	"New York"
]
['Boston', 'Chicago', 'Los Angles', 'Miami', 'New York']


# A General Example

In [57]:
print("\n" + "=" * 50)
print("REAL LIFE EXAMPLE: Employee Analysis")
print("=" * 50)

# Detailed analysis (expressions inside the same with_columns can't see each other's new columns,
# so Net Salary is created in a second, chained with_columns)
df = df.with_columns(
    (pl.col("Salary") * 0.15).alias("Tax"),
    pl.Series("Performance", ["A", "B", "A", "C", "B"]),
).with_columns((pl.col("Salary") - pl.col("Tax")).alias("Net Salary"))

print("Detailed Employee DataFrame:")
print(df)

# Detailed analysis
analysis = (
    df.group_by("Performance")
    .agg(
        pl.col("Salary").count().alias("Salary_count"),
        pl.col("Salary").mean().alias("Salary_mean"),
        pl.col("Salary").std().alias("Salary_std"),
        pl.col("Age").min().alias("Age_min"),
        pl.col("Age").max().alias("Age_max"),
        pl.col("Age").mean().alias("Age_mean"),
        pl.col("Net Salary").sum().alias("Net Salary_sum"),
    )
    .sort("Performance")
    .with_columns(cs.float().round(2))
)

print("\nPerformance Analysis:")
print(analysis)

# Correlation analysis (cs.numeric() selects all numeric columns)
numeric_df = df.select(cs.numeric())
print("\nCorrelation Matrix:")
print(numeric_df.corr())


REAL LIFE EXAMPLE: Employee Analysis
Detailed Employee DataFrame:
shape: (5, 11)
┌───────┬─────┬────────────┬────────┬───┬────────┬─────────┬─────────────┬────────────┐
│ Name  ┆ Age ┆ City       ┆ Salary ┆ … ┆ Gender ┆ Tax     ┆ Performance ┆ Net Salary │
│ ---   ┆ --- ┆ ---        ┆ ---    ┆   ┆ ---    ┆ ---     ┆ ---         ┆ ---        │
│ str   ┆ i64 ┆ cat        ┆ i64    ┆   ┆ str    ┆ f64     ┆ str         ┆ f64        │
╞═══════╪═════╪════════════╪════════╪═══╪════════╪═════════╪═════════════╪════════════╡
│ Jack  ┆ 25  ┆ New York   ┆ 50000  ┆ … ┆ M      ┆ 7500.0  ┆ A           ┆ 42500.0    │
│ Jim   ┆ 30  ┆ Boston     ┆ 60000  ┆ … ┆ M      ┆ 9000.0  ┆ B           ┆ 51000.0    │
│ John  ┆ 35  ┆ Los Angles ┆ 70000  ┆ … ┆ M      ┆ 10500.0 ┆ A           ┆ 59500.0    │
│ Jill  ┆ 28  ┆ Miami      ┆ 55000  ┆ … ┆ F      ┆ 8250.0  ┆ C           ┆ 46750.0    │
│ Jenny ┆ 32  ┆ Chicago    ┆ 65000  ┆ … ┆ F      ┆ 9750.0  ┆ B           ┆ 55250.0    │
└───────┴─────┴────────────┴────────┴─

# **Important Features of DataFrame:**
2-Dimensional: Consists of rows and columns

Flexible Data Types: Can hold data of different types (strictly typed per column)

Named Access: Named columns, rows addressed by position (no index)

Missing Data Support: Can manage null values

Powerful Selection: expressions (`pl.col`, selectors), `filter`, positional slicing

Grouping: Powerful aggregation with group_by

Time Series: Native Date/Datetime/Duration types, `group_by_dynamic`, `rolling`

Lazy Evaluation: LazyFrame optimizes the whole query before running it

DataFrame is an indispensable tool for data analysis, machine learning, and data cleaning operations!

# head() glimpse() describe()

Most common methods used to understand DataFrame
(`glimpse()` is the Polars counterpart of Pandas' `info()`)

# head() method

In [58]:
# Let's create an example DataFrame
data = {
    "Name": ["Ali", "Ayşe", "Mehmet", "Zeynep", "Can", "Elif", "Burak", "Deniz"],
    "Age": [25, 30, 35, 28, 32, 29, 41, 26],
    "City": [
        "Istanbul",
        "Ankara",
        "Izmir",
        "Bursa",
        "Antalya",
        "Istanbul",
        "Ankara",
        "Izmir",
    ],
    "Salary": [50000, 60000, 70000, 55000, 65000, 58000, 80000, 52000],
    "Department": ["IT", "HR", "IT", "Finance", "IT", "HR", "IT", "Finance"],
    "Experience": [2, 5, 8, 3, 6, 4, 12, 3],
}

df = pl.DataFrame(data)
print("Full DataFrame:")
print(df)
print(f"DataFrame size: {df.shape}")  # (8, 6)

Full DataFrame:
shape: (8, 6)
┌────────┬─────┬──────────┬────────┬────────────┬────────────┐
│ Name   ┆ Age ┆ City     ┆ Salary ┆ Department ┆ Experience │
│ ---    ┆ --- ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str    ┆ i64 ┆ str      ┆ i64    ┆ str        ┆ i64        │
╞════════╪═════╪══════════╪════════╪════════════╪════════════╡
│ Ali    ┆ 25  ┆ Istanbul ┆ 50000  ┆ IT         ┆ 2          │
│ Ayşe   ┆ 30  ┆ Ankara   ┆ 60000  ┆ HR         ┆ 5          │
│ Mehmet ┆ 35  ┆ Izmir    ┆ 70000  ┆ IT         ┆ 8          │
│ Zeynep ┆ 28  ┆ Bursa    ┆ 55000  ┆ Finance    ┆ 3          │
│ Can    ┆ 32  ┆ Antalya  ┆ 65000  ┆ IT         ┆ 6          │
│ Elif   ┆ 29  ┆ Istanbul ┆ 58000  ┆ HR         ┆ 4          │
│ Burak  ┆ 41  ┆ Ankara   ┆ 80000  ┆ IT         ┆ 12         │
│ Deniz  ┆ 26  ┆ Izmir    ┆ 52000  ┆ Finance    ┆ 3          │
└────────┴─────┴──────────┴────────┴────────────┴────────────┘
DataFrame size: (8, 6)


In [59]:
# head method usage

print("\n" + "=" * 50)
print("head() METHOD")
print("=" * 50)

# Default: first 5 rows (DataFrame.head default is 5; Series.head default is 10)
print("df.head() - First 5 rows:")
print(df.head())

# Specific number of rows
print("\ndf.head(3) - First 3 rows:")
print(df.head(3))

print("\ndf.head(7) - First 7 rows:")
print(df.head(7))

# Negative number (except last n rows)
print("\ndf.head(-2) - Except last 2 rows:")
print(df.head(-2))


head() METHOD
df.head() - First 5 rows:
shape: (5, 6)
┌────────┬─────┬──────────┬────────┬────────────┬────────────┐
│ Name   ┆ Age ┆ City     ┆ Salary ┆ Department ┆ Experience │
│ ---    ┆ --- ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str    ┆ i64 ┆ str      ┆ i64    ┆ str        ┆ i64        │
╞════════╪═════╪══════════╪════════╪════════════╪════════════╡
│ Ali    ┆ 25  ┆ Istanbul ┆ 50000  ┆ IT         ┆ 2          │
│ Ayşe   ┆ 30  ┆ Ankara   ┆ 60000  ┆ HR         ┆ 5          │
│ Mehmet ┆ 35  ┆ Izmir    ┆ 70000  ┆ IT         ┆ 8          │
│ Zeynep ┆ 28  ┆ Bursa    ┆ 55000  ┆ Finance    ┆ 3          │
│ Can    ┆ 32  ┆ Antalya  ┆ 65000  ┆ IT         ┆ 6          │
└────────┴─────┴──────────┴────────┴────────────┴────────────┘

df.head(3) - First 3 rows:
shape: (3, 6)
┌────────┬─────┬──────────┬────────┬────────────┬────────────┐
│ Name   ┆ Age ┆ City     ┆ Salary ┆ Department ┆ Experience │
│ ---    ┆ --- ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str    ┆ i64 ┆ str 

In [60]:
# getting info about dataframe with glimpse()
print("\n" + "=" * 50)
print("glimpse() METHOD")
print("=" * 50)

# Basic info: one line per column with dtype and the first values
print("df.glimpse() - Basic information:")
df.glimpse()

# Short info: only column names and dtypes
print("\ndf.schema - Short info:")
print(df.schema)

# Memory usage (Polars estimates it from the Arrow buffers)
print("\ndf.estimated_size() - Memory usage:")
print(f"{df.estimated_size('b')} bytes ({df.estimated_size('kb'):.2f} KB)")

# Missing data per column (the "Non-Null Count" part of Pandas' info())
print("\ndf.null_count() - Missing values per column:")
print(df.null_count())


glimpse() METHOD
df.glimpse() - Basic information:
Rows: 8
Columns: 6
$ Name       <str> 'Ali', 'Ayşe', 'Mehmet', 'Zeynep', 'Can', 'Elif', 'Burak', 'Deniz'
$ Age        <i64> 25, 30, 35, 28, 32, 29, 41, 26
$ City       <str> 'Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya', 'Istanbul', 'Ankara', 'Izmir'
$ Salary     <i64> 50000, 60000, 70000, 55000, 65000, 58000, 80000, 52000
$ Department <str> 'IT', 'HR', 'IT', 'Finance', 'IT', 'HR', 'IT', 'Finance'
$ Experience <i64> 2, 5, 8, 3, 6, 4, 12, 3


df.schema - Short info:
Schema({'Name': String, 'Age': Int64, 'City': String, 'Salary': Int64, 'Department': String, 'Experience': Int64})

df.estimated_size() - Memory usage:
305 bytes (0.30 KB)

df.null_count() - Missing values per column:
shape: (1, 6)
┌──────┬─────┬──────┬────────┬────────────┬────────────┐
│ Name ┆ Age ┆ City ┆ Salary ┆ Department ┆ Experience │
│ ---  ┆ --- ┆ ---  ┆ ---    ┆ ---        ┆ ---        │
│ u32  ┆ u32 ┆ u32  ┆ u32    ┆ u32        ┆ u32        │
╞══════╪═════╪

In [61]:
# to better understand glimpse output
glimpse_text = df.glimpse(return_type="string")
print(glimpse_text)

print("\nglimpse() OUTPUT ANALYSIS:")
print("• Rows: 8 → number of rows (there is no RangeIndex in Polars)")
print("• Columns: 6 → Total 6 columns")
print("• Each following line describes one column:")
print("  - Column name")
print("  - Data type (<i64> integer, <str> string, <f64> float, ...)")
print("  - First few values")
print("• Non-null counts → df.null_count()")
print("• Memory usage → df.estimated_size()")

Rows: 8
Columns: 6
$ Name       <str> 'Ali', 'Ayşe', 'Mehmet', 'Zeynep', 'Can', 'Elif', 'Burak', 'Deniz'
$ Age        <i64> 25, 30, 35, 28, 32, 29, 41, 26
$ City       <str> 'Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya', 'Istanbul', 'Ankara', 'Izmir'
$ Salary     <i64> 50000, 60000, 70000, 55000, 65000, 58000, 80000, 52000
$ Department <str> 'IT', 'HR', 'IT', 'Finance', 'IT', 'HR', 'IT', 'Finance'
$ Experience <i64> 2, 5, 8, 3, 6, 4, 12, 3


glimpse() OUTPUT ANALYSIS:
• Rows: 8 → number of rows (there is no RangeIndex in Polars)
• Columns: 6 → Total 6 columns
• Each following line describes one column:
  - Column name
  - Data type (<i64> integer, <str> string, <f64> float, ...)
  - First few values
• Non-null counts → df.null_count()
• Memory usage → df.estimated_size()


In [62]:
# describe() - statistical summary

print("\n" + "=" * 50)
print("describe() METHOD")
print("=" * 50)

# describe for entire DataFrame (Polars describes ALL columns by default,
# there is no include='all' option)
print("df.describe() - Statistics for all columns:")
print(df.describe())

# Only numeric columns: select them first
print("\ndf.select(cs.numeric()).describe() - Numeric columns only:")
print(df.select(cs.numeric()).describe())


describe() METHOD
df.describe() - Statistics for all columns:
shape: (9, 7)
┌────────────┬────────┬──────────┬────────┬──────────────┬────────────┬────────────┐
│ statistic  ┆ Name   ┆ Age      ┆ City   ┆ Salary       ┆ Department ┆ Experience │
│ ---        ┆ ---    ┆ ---      ┆ ---    ┆ ---          ┆ ---        ┆ ---        │
│ str        ┆ str    ┆ f64      ┆ str    ┆ f64          ┆ str        ┆ f64        │
╞════════════╪════════╪══════════╪════════╪══════════════╪════════════╪════════════╡
│ count      ┆ 8      ┆ 8.0      ┆ 8      ┆ 8.0          ┆ 8          ┆ 8.0        │
│ null_count ┆ 0      ┆ 0.0      ┆ 0      ┆ 0.0          ┆ 0          ┆ 0.0        │
│ mean       ┆ null   ┆ 30.75    ┆ null   ┆ 61250.0      ┆ null       ┆ 5.375      │
│ std        ┆ null   ┆ 5.230406 ┆ null   ┆ 10039.208848 ┆ null       ┆ 3.292307   │
│ min        ┆ Ali    ┆ 25.0     ┆ Ankara ┆ 50000.0      ┆ Finance    ┆ 2.0        │
│ 25%        ┆ null   ┆ 28.0     ┆ null   ┆ 55000.0      ┆ null       ┆ 3

In [63]:
print("\nDESCRIBE OUTPUT ANALYSIS:")
describe_df = df.describe()
print(
    "The first column ('statistic') names the statistic, the other columns hold the values:"
)
print("• count:      Number of non-null records")
print("• null_count: Number of missing values")
print("• mean:       Average value")
print("• std:        Standard deviation (measure of variability)")
print("• min:        Minimum value")
print("• 25%:        First quartile (Q1)")
print("• 50%:        Median (second quartile)")
print("• 75%:        Third quartile (Q3)")
print("• max:        Maximum value")
print("(for string columns only count, null_count, min and max are meaningful)")
print(
    "\nNote: describe() computes 25%/50%/75% with 'nearest' interpolation, Pandas uses 'linear'."
)
print(
    f"  Age Q1 -> describe/nearest: {df['Age'].quantile(0.25, interpolation='nearest')}, "
    f"linear: {df['Age'].quantile(0.25, interpolation='linear')}"
)

print(f"\nExample Interpretation - Age column:")
print(f"• Average age: {df['Age'].mean():.1f}")
print(f"• Youngest: {df['Age'].min()}, Oldest: {df['Age'].max()}")
print(f"• Age distribution: {df['Age'].std():.1f} standard deviation")


DESCRIBE OUTPUT ANALYSIS:
The first column ('statistic') names the statistic, the other columns hold the values:
• count:      Number of non-null records
• null_count: Number of missing values
• mean:       Average value
• std:        Standard deviation (measure of variability)
• min:        Minimum value
• 25%:        First quartile (Q1)
• 50%:        Median (second quartile)
• 75%:        Third quartile (Q3)
• max:        Maximum value
(for string columns only count, null_count, min and max are meaningful)

Note: describe() computes 25%/50%/75% with 'nearest' interpolation, Pandas uses 'linear'.
  Age Q1 -> describe/nearest: 28.0, linear: 27.5

Example Interpretation - Age column:
• Average age: 30.8
• Youngest: 25, Oldest: 41
• Age distribution: 5.2 standard deviation


# Usage with Different Data Types for head glimpse describe

In [64]:
print("\n" + "=" * 50)
print("USAGE WITH DIFFERENT DATA TYPES")
print("=" * 50)

# Complex DataFrame
mixed_df = pl.DataFrame(
    {
        "numeric": [1, 2, 3, 4, 5, 6, 7, 8],
        "text": ["A", "B", "C", "D", "E", "F", "G", "H"],
        "category": pl.Series(
            ["X", "Y", "X", "Y", "X", "Y", "X", "Y"], dtype=pl.Categorical
        ),
        "date": pl.date_range(
            date(2023, 1, 1), date(2023, 1, 8), interval="1d", eager=True
        ),
        "bool": [True, False, True, False, True, False, True, False],
        "decimal": [1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8],
    }
)

print("Complex DataFrame:")
print(mixed_df.head())

print("\nglimpse():")
mixed_df.glimpse()

print("\ndescribe():")
print(mixed_df.describe())


USAGE WITH DIFFERENT DATA TYPES
Complex DataFrame:
shape: (5, 6)
┌─────────┬──────┬──────────┬────────────┬───────┬─────────┐
│ numeric ┆ text ┆ category ┆ date       ┆ bool  ┆ decimal │
│ ---     ┆ ---  ┆ ---      ┆ ---        ┆ ---   ┆ ---     │
│ i64     ┆ str  ┆ cat      ┆ date       ┆ bool  ┆ f64     │
╞═════════╪══════╪══════════╪════════════╪═══════╪═════════╡
│ 1       ┆ A    ┆ X        ┆ 2023-01-01 ┆ true  ┆ 1.1     │
│ 2       ┆ B    ┆ Y        ┆ 2023-01-02 ┆ false ┆ 2.2     │
│ 3       ┆ C    ┆ X        ┆ 2023-01-03 ┆ true  ┆ 3.3     │
│ 4       ┆ D    ┆ Y        ┆ 2023-01-04 ┆ false ┆ 4.4     │
│ 5       ┆ E    ┆ X        ┆ 2023-01-05 ┆ true  ┆ 5.5     │
└─────────┴──────┴──────────┴────────────┴───────┴─────────┘

glimpse():
Rows: 8
Columns: 6
$ numeric   <i64> 1, 2, 3, 4, 5, 6, 7, 8
$ text      <str> 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'
$ category  <cat> X, Y, X, Y, X, Y, X, Y
$ date     <date> 2023-01-01, 2023-01-02, 2023-01-03, 2023-01-04, 2023-01-05, 2023-01-06, 202

# Usage with Missing Data

In [65]:
print("\n" + "=" * 50)
print("USAGE WITH MISSING DATA")
print("=" * 50)

# DataFrame with missing data
df_missing = df.with_columns(
    pl.when(pl.int_range(pl.len()) == 0)
    .then(None)
    .otherwise(pl.col("Age"))
    .alias("Age"),
    pl.when(pl.int_range(pl.len()) == 2)
    .then(None)
    .otherwise(pl.col("Salary"))
    .alias("Salary"),
    pl.when(pl.int_range(pl.len()) == 4)
    .then(None)
    .otherwise(pl.col("Name"))
    .alias("Name"),
)

print("DataFrame with missing data:")
print(df_missing.head())

print("\nnull_count() - shows missing data:")
print(df_missing.null_count())

print("\ndescribe() - count excludes missing data, null_count reports it:")
print(df_missing.describe())


USAGE WITH MISSING DATA
DataFrame with missing data:
shape: (5, 6)
┌────────┬──────┬──────────┬────────┬────────────┬────────────┐
│ Name   ┆ Age  ┆ City     ┆ Salary ┆ Department ┆ Experience │
│ ---    ┆ ---  ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str    ┆ i64  ┆ str      ┆ i64    ┆ str        ┆ i64        │
╞════════╪══════╪══════════╪════════╪════════════╪════════════╡
│ Ali    ┆ null ┆ Istanbul ┆ 50000  ┆ IT         ┆ 2          │
│ Ayşe   ┆ 30   ┆ Ankara   ┆ 60000  ┆ HR         ┆ 5          │
│ Mehmet ┆ 35   ┆ Izmir    ┆ null   ┆ IT         ┆ 8          │
│ Zeynep ┆ 28   ┆ Bursa    ┆ 55000  ┆ Finance    ┆ 3          │
│ null   ┆ 32   ┆ Antalya  ┆ 65000  ┆ IT         ┆ 6          │
└────────┴──────┴──────────┴────────┴────────────┴────────────┘

null_count() - shows missing data:
shape: (1, 6)
┌──────┬─────┬──────┬────────┬────────────┬────────────┐
│ Name ┆ Age ┆ City ┆ Salary ┆ Department ┆ Experience │
│ ---  ┆ --- ┆ ---  ┆ ---    ┆ ---        ┆ ---        │
│ u32  

Practical Usage Scenarios

In [66]:
# scenario 1 : quick data exploration
print("\n" + "=" * 50)
print("SCENARIO 1: QUICK DATA EXPLORATION")
print("=" * 50)


def quick_exploration(dataframe):
    """To quickly understand DataFrame"""
    print("📊 QUICK DATA EXPLORATION")
    print("=" * 30)

    print("1. FIRST LOOK (head):")
    print(dataframe.head(3))

    print("\n2. BASIC INFO (glimpse):")
    dataframe.glimpse()

    print("\n3. STATISTICS (describe):")
    print(dataframe.describe())

    print("\n4. SIZE:")
    print(f"   {dataframe.height} rows, {dataframe.width} columns")

    print("\n5. MISSING DATA:")
    print(f"   Total {dataframe.null_count().sum_horizontal().item()} missing data")


# Apply quick exploration
quick_exploration(df)


SCENARIO 1: QUICK DATA EXPLORATION
📊 QUICK DATA EXPLORATION
1. FIRST LOOK (head):
shape: (3, 6)
┌────────┬─────┬──────────┬────────┬────────────┬────────────┐
│ Name   ┆ Age ┆ City     ┆ Salary ┆ Department ┆ Experience │
│ ---    ┆ --- ┆ ---      ┆ ---    ┆ ---        ┆ ---        │
│ str    ┆ i64 ┆ str      ┆ i64    ┆ str        ┆ i64        │
╞════════╪═════╪══════════╪════════╪════════════╪════════════╡
│ Ali    ┆ 25  ┆ Istanbul ┆ 50000  ┆ IT         ┆ 2          │
│ Ayşe   ┆ 30  ┆ Ankara   ┆ 60000  ┆ HR         ┆ 5          │
│ Mehmet ┆ 35  ┆ Izmir    ┆ 70000  ┆ IT         ┆ 8          │
└────────┴─────┴──────────┴────────┴────────────┴────────────┘

2. BASIC INFO (glimpse):
Rows: 8
Columns: 6
$ Name       <str> 'Ali', 'Ayşe', 'Mehmet', 'Zeynep', 'Can', 'Elif', 'Burak', 'Deniz'
$ Age        <i64> 25, 30, 35, 28, 32, 29, 41, 26
$ City       <str> 'Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya', 'Istanbul', 'Ankara', 'Izmir'
$ Salary     <i64> 50000, 60000, 70000, 55000, 65000, 5

In [67]:
# scenario 2 : data quality control

print("\n" + "=" * 50)
print("SCENARIO 2: DATA QUALITY CONTROL")
print("=" * 50)


def data_quality_control(dataframe):
    """Check data quality"""
    print("✅ DATA QUALITY REPORT")
    print("=" * 35)

    # Size info
    print(f"Row count: {len(dataframe)}")
    print(f"Column count: {len(dataframe.columns)}")

    # Missing data analysis
    missing_data = dataframe.null_count().row(0, named=True)
    print(f"\nMISSING DATA ANALYSIS:")
    for column, missing in missing_data.items():
        if missing > 0:
            print(f"  ⚠️  {column}: {missing} missing data")
        else:
            print(f"  ✅ {column}: No missing data")

    # Data types
    print(f"\nDATA TYPES:")
    for column, tip in dataframe.schema.items():
        print(f"  {column}: {tip}")

    # Unique values
    print(f"\nUNIQUE VALUE COUNTS:")
    for column in dataframe.columns:
        unique = dataframe[column].n_unique()
        print(f"  {column}: {unique} unique values")


data_quality_control(df)


SCENARIO 2: DATA QUALITY CONTROL
✅ DATA QUALITY REPORT
Row count: 8
Column count: 6

MISSING DATA ANALYSIS:
  ✅ Name: No missing data
  ✅ Age: No missing data
  ✅ City: No missing data
  ✅ Salary: No missing data
  ✅ Department: No missing data
  ✅ Experience: No missing data

DATA TYPES:
  Name: String
  Age: Int64
  City: String
  Salary: Int64
  Department: String
  Experience: Int64

UNIQUE VALUE COUNTS:
  Name: 8 unique values
  Age: 8 unique values
  City: 5 unique values
  Salary: 8 unique values
  Department: 3 unique values
  Experience: 7 unique values


In [68]:
# Scenario 3: Analysis by Groups

print("\n" + "=" * 50)
print("SCENARIO 3: ANALYSIS BY GROUPS")
print("=" * 50)

# Analysis by department
print("SUMMARY BY DEPARTMENT:")

for department in df["Department"].unique(maintain_order=True):
    print(f"\n--- {department} Department ---")
    department_df = df.filter(pl.col("Department") == department)
    print(department_df.describe())


SCENARIO 3: ANALYSIS BY GROUPS
SUMMARY BY DEPARTMENT:

--- IT Department ---
shape: (9, 7)
┌────────────┬────────┬──────────┬────────┬─────────┬────────────┬────────────┐
│ statistic  ┆ Name   ┆ Age      ┆ City   ┆ Salary  ┆ Department ┆ Experience │
│ ---        ┆ ---    ┆ ---      ┆ ---    ┆ ---     ┆ ---        ┆ ---        │
│ str        ┆ str    ┆ f64      ┆ str    ┆ f64     ┆ str        ┆ f64        │
╞════════════╪════════╪══════════╪════════╪═════════╪════════════╪════════════╡
│ count      ┆ 4      ┆ 4.0      ┆ 4      ┆ 4.0     ┆ 4          ┆ 4.0        │
│ null_count ┆ 0      ┆ 0.0      ┆ 0      ┆ 0.0     ┆ 0          ┆ 0.0        │
│ mean       ┆ null   ┆ 33.25    ┆ null   ┆ 66250.0 ┆ null       ┆ 7.0        │
│ std        ┆ null   ┆ 6.652067 ┆ null   ┆ 12500.0 ┆ null       ┆ 4.163332   │
│ min        ┆ Ali    ┆ 25.0     ┆ Ankara ┆ 50000.0 ┆ IT         ┆ 2.0        │
│ 25%        ┆ null   ┆ 32.0     ┆ null   ┆ 65000.0 ┆ null       ┆ 6.0        │
│ 50%        ┆ null   ┆ 35.0

In [69]:
print("\n" + "=" * 50)
print("SPECIAL STATISTICS")
print("=" * 50)

# Percentiles parameter
print("describe with custom percentiles:")
print(df.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

# Date columns are supported natively (no datetime_is_numeric flag needed)
print("\ndescribe with date columns:")
date_df = pl.DataFrame(
    {
        "date": pl.date_range(
            date(2023, 1, 1), date(2023, 1, 10), interval="1d", eager=True
        ),
        "value": range(10),
    }
)
print(date_df.describe())


SPECIAL STATISTICS
describe with custom percentiles:
shape: (11, 7)
┌────────────┬────────┬──────────┬────────┬──────────────┬────────────┬────────────┐
│ statistic  ┆ Name   ┆ Age      ┆ City   ┆ Salary       ┆ Department ┆ Experience │
│ ---        ┆ ---    ┆ ---      ┆ ---    ┆ ---          ┆ ---        ┆ ---        │
│ str        ┆ str    ┆ f64      ┆ str    ┆ f64          ┆ str        ┆ f64        │
╞════════════╪════════╪══════════╪════════╪══════════════╪════════════╪════════════╡
│ count      ┆ 8      ┆ 8.0      ┆ 8      ┆ 8.0          ┆ 8          ┆ 8.0        │
│ null_count ┆ 0      ┆ 0.0      ┆ 0      ┆ 0.0          ┆ 0          ┆ 0.0        │
│ mean       ┆ null   ┆ 30.75    ┆ null   ┆ 61250.0      ┆ null       ┆ 5.375      │
│ std        ┆ null   ┆ 5.230406 ┆ null   ┆ 10039.208848 ┆ null       ┆ 3.292307   │
│ min        ┆ Ali    ┆ 25.0     ┆ Ankara ┆ 50000.0      ┆ Finance    ┆ 2.0        │
│ …          ┆ …      ┆ …        ┆ …      ┆ …            ┆ …          ┆ …        

In [70]:
# customizing outputs
print("\n" + "=" * 50)
print("CUSTOMIZING OUTPUTS")
print("=" * 50)


# More readable outputs
def nice_report(dataframe):
    print("📈 DETAILED DATA FRAME REPORT")
    print("=" * 40)

    # Head with border (pl.Config controls how tables are displayed)
    print("\nFIRST 4 ROWS:")
    with pl.Config(tbl_formatting="ASCII_FULL", tbl_hide_dataframe_shape=True):
        print(dataframe.head(4))

    # Info summary
    print(f"\n📋 GENERAL INFO:")
    print(f"   • Shape: {dataframe.height} rows × {dataframe.width} columns")
    print(f"   • Memory: {dataframe.estimated_size('kb'):.1f} KB")

    # Describe with formatting
    print(f"\n📊 STATISTICS:")
    stats = dataframe.describe().with_columns(cs.float().round(2))
    print(stats)


nice_report(df)


CUSTOMIZING OUTPUTS
📈 DETAILED DATA FRAME REPORT

FIRST 4 ROWS:
+--------+-----+----------+--------+------------+------------+
| Name   | Age | City     | Salary | Department | Experience |
| ---    | --- | ---      | ---    | ---        | ---        |
| str    | i64 | str      | i64    | str        | i64        |
+============================================================+
| Ali    | 25  | Istanbul | 50000  | IT         | 2          |
|--------+-----+----------+--------+------------+------------|
| Ayşe   | 30  | Ankara   | 60000  | HR         | 5          |
|--------+-----+----------+--------+------------+------------|
| Mehmet | 35  | Izmir    | 70000  | IT         | 8          |
|--------+-----+----------+--------+------------+------------|
| Zeynep | 28  | Bursa    | 55000  | Finance    | 3          |
+--------+-----+----------+--------+------------+------------+

📋 GENERAL INFO:
   • Shape: 8 rows × 6 columns
   • Memory: 0.3 KB

📊 STATISTICS:
shape: (9, 7)
┌────────────┬─────